In [1]:
# Cell 1 - Environment Setup (revised)
import os, sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# PYTHONHASHSEED: set dari shell sebelum launch kernel, bukan di sini.

import warnings
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

import gc, random, logging
import numpy as np
import torch
import tensorflow as tf

logging.getLogger("absl").setLevel(logging.ERROR)
tf.get_logger().setLevel("ERROR")

SEED = 42

# --- TF GPU config HARUS sebelum GPU dipakai ---
tf_memory_growth = False
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        tf_memory_growth = True
    except RuntimeError as e:
        print(f"tf_memory_growth_error: {e}")

# --- Seeds ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if sys.platform in ("win32", "darwin") else min(4, os.cpu_count() or 2)

# --- Determinism (berlaku untuk CPU & GPU) ---
torch.use_deterministic_algorithms(True, warn_only=True)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    free_vram, total_vram = torch.cuda.mem_get_info(DEVICE)
    print(f"hardware_target | device={DEVICE} | gpu={torch.cuda.get_device_name(DEVICE)} "
          f"| vram_free={free_vram/1024**3:.2f}GB | total={total_vram/1024**3:.2f}GB "
          f"| tf_memory_growth={tf_memory_growth}")
else:
    print(f"hardware_target | device={DEVICE} (CPU) | tf_memory_growth={tf_memory_growth}")

print(f"versions | torch={torch.__version__} | tf={tf.__version__} | cuda={torch.version.cuda}")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_generator(seed=SEED):
    """Buat generator baru per DataLoader agar urutan shuffle reproducible."""
    gen = torch.Generator()
    gen.manual_seed(seed)
    return gen

def purge_vram():
    """Panggil SETELAH `del` model/optimizer/tensor besar."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print(f"environment_ready | seed={SEED} | workers={NUM_WORKERS}")

hardware_target | device=cuda:0 | gpu=Tesla T4 | vram_free=14.46GB | total=14.56GB | tf_memory_growth=True
versions | torch=2.11.0+cu128 | tf=2.20.0 | cuda=12.8
environment_ready | seed=42 | workers=4


In [2]:
# Cell 2 - Remote Dataset Discovery & Self-Contained Combinatorial 4-Fold Generator (revised)
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd

# --- Konstanta ---
N_FOLDS = 4
N_TRAIN, N_TEST = 1826, 814
N_BOND_GROUPS = 7
EXPECTED_FOLD_CENDRA = [35, 29, 16, 3]

def check(cond, msg):
    if not cond:
        raise ValueError(msg)

candidate_base_dirs = [
    Path("/kaggle/input/competitions/schematics-2026"),
    Path("/kaggle/input/schematics-2026"),
    Path("schematics-2026"),
    Path("C:/Github/schematics-npc-dsc-2026/schematics-2026"),
]

base_dir = next((p for p in candidate_base_dirs if p.exists()), None)
if base_dir is None:
    raise FileNotFoundError(f"Dataset schematics-2026 tidak ditemukan. Dicoba: {[str(p) for p in candidate_base_dirs]}")

WORK_DIR = Path("/kaggle/working") if str(base_dir).startswith("/kaggle") else Path("schematics_day5_all_artifacts")
WORK_DIR.mkdir(parents=True, exist_ok=True)

train_meta_path = base_dir / "train" / "metadata.csv"
sub_path = base_dir / "sample_submission.csv"

train_df = pd.read_csv(train_meta_path, dtype={"file_id": str})
sub_df = pd.read_csv(sub_path, dtype={"file_id": str})

check(train_df["label"].notna().all(), "Ada label NaN di metadata train")
train_df["label"] = train_df["label"].str.strip()
train_audio_dir = base_dir / "train" / "audio"
test_audio_dir = base_dir / "test" / "audio"

train_df["filepath"] = [str(train_audio_dir / f"{fid}.flac") for fid in train_df["file_id"]]
sub_df["filepath"] = [str(test_audio_dir / f"{fid}.flac") for fid in sub_df["file_id"]]

check(len(train_df) == N_TRAIN and len(sub_df) == N_TEST, "Jumlah baris data tidak sesuai")
check(train_df["file_id"].nunique() == N_TRAIN and sub_df["file_id"].nunique() == N_TEST, "file_id duplikat")
check(train_df["source_group"].notna().all(), "Ada source_group kosong (NaN)")
check((train_df.groupby("source_group")["label"].nunique() == 1).all(), "Ada source_group dengan >1 label")

missing_train = [p for p in train_df["filepath"] if not Path(p).exists()]
missing_test = [p for p in sub_df["filepath"] if not Path(p).exists()]
check(not missing_train and not missing_test,
      f"File audio hilang | train={len(missing_train)} (contoh: {missing_train[:2]}) | "
      f"test={len(missing_test)} (contoh: {missing_test[:2]})")

folds_meta_path = WORK_DIR / "metadata_with_folds.csv"

def load_cached_folds(path, df, n_folds):
    if not path.exists():
        return None
    try:
        saved = pd.read_csv(path, dtype={"file_id": str, "source_group": str})
        if not {"file_id", "fold", "source_group"}.issubset(saved.columns):
            return None
        saved = saved.drop_duplicates("file_id").set_index("file_id")
        if set(saved.index) != set(df["file_id"]):
            return None
        folds = df["file_id"].map(saved["fold"])
        if folds.isna().any() or not folds.isin(range(n_folds)).all():
            return None
        same_group = (df["file_id"].map(saved["source_group"]).values
                      == df["source_group"].astype(str).values).all()
        if not same_group:
            return None
        return folds.astype(int)
    except Exception as e:
        print(f"cache_invalid: {e}")
        return None

cached = load_cached_folds(folds_meta_path, train_df, N_FOLDS)
if cached is not None:
    train_df["fold"] = cached
    fold_source = "cache"
else:
    fold_source = "regenerated"
    grp_meta = (
        train_df.groupby(["label", "source_group"])
        .size()
        .reset_index(name="n_samples")
        .sort_values(["label", "n_samples", "source_group"], ascending=[True, False, True])
    )
    cend_grps = grp_meta[grp_meta["label"] == "Cendrawasih"][["source_group", "n_samples"]].values.tolist()
    kuti_grps = grp_meta[grp_meta["label"] == "Kutilang"][["source_group", "n_samples"]].values.tolist()
    bond_grps = grp_meta[grp_meta["label"] == "Bondol Jawa"][["source_group", "n_samples"]].values.tolist()

    check(len(cend_grps) == N_FOLDS and len(kuti_grps) == N_FOLDS and len(bond_grps) == N_BOND_GROUPS,
          f"Jumlah grup tidak sesuai | cend={len(cend_grps)} kuti={len(kuti_grps)} bond={len(bond_grps)}")

    group_to_fold = {}
    for f in range(N_FOLDS):
        group_to_fold[cend_grps[f][0]] = f
        group_to_fold[kuti_grps[N_FOLDS - 1 - f][0]] = f

    base_tot = [cend_grps[f][1] + kuti_grps[N_FOLDS - 1 - f][1] for f in range(N_FOLDS)]

    best_score = float("inf")
    best_bond_assign = None
    for assign in product(range(N_FOLDS), repeat=len(bond_grps)):
        if len(set(assign)) < N_FOLDS:
            continue
        fold_bond = [0] * N_FOLDS
        fold_tot = list(base_tot)
        for (_, sz), f in zip(bond_grps, assign):
            fold_bond[f] += sz
            fold_tot[f] += sz
        score = np.std(fold_bond) + 0.5 * np.std(fold_tot)
        if score < best_score:
            best_score = score
            best_bond_assign = assign

    for (grp_id, _), f in zip(bond_grps, best_bond_assign):
        group_to_fold[grp_id] = f

    train_df["fold"] = train_df["source_group"].map(group_to_fold)
    check(train_df["fold"].notna().all(), "Ada source_group yang tidak ter-assign ke fold")
    train_df["fold"] = train_df["fold"].astype(int)
    train_df.drop(columns=["filepath"]).to_csv(folds_meta_path, index=False)

val_groups = [set(train_df[train_df["fold"] == f]["source_group"].unique()) for f in range(N_FOLDS)]
leakage = sum(len(val_groups[i] & val_groups[j]) for i in range(N_FOLDS) for j in range(i + 1, N_FOLDS))
check(leakage == 0, f"Group leakage terdeteksi: {leakage}")

fold_cendra = [int((train_df[train_df["fold"] == f]["label"] == "Cendrawasih").sum()) for f in range(N_FOLDS)]
check(fold_cendra == EXPECTED_FOLD_CENDRA, f"Distribusi Cendrawasih menyimpang: {fold_cendra}")

print(pd.crosstab(train_df["fold"], train_df["label"]))
print(f"metadata_locked | fold_source={fold_source} | train={len(train_df)} | test={len(sub_df)} | "
      f"fold_cendra={fold_cendra} | group_leakage={leakage}")

label  Bondol Jawa  Cendrawasih  Kutilang
fold                                     
0              336           35       132
1              290           29       143
2              240           16       166
3              258            3       178
metadata_locked | fold_source=regenerated | train=1826 | test=814 | fold_cendra=[35, 29, 16, 3] | group_leakage=0


In [4]:
# Cell 3 - Fast Dual-Stream DSP RAM Pre-Caching (32 kHz 5s & 48 kHz 3s) (revised v2)
from math import gcd
import os
import time
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly, butter, sosfilt
from joblib import Parallel, delayed

t0_dsp = time.time()
DSP_JOBS = max(1, min(4, os.cpu_count() or 2))   # terpisah dari NUM_WORKERS (DataLoader)
SILENT_EPS = 1e-4

# Filter dihitung sekali; array numpy aman di-pickle ke worker loky
SOS_HP_100 = butter(4, 100.0, btype="highpass", fs=32000, output="sos")
SOS_BP_VAD = butter(2, [500.0, 8000.0], btype="bandpass", fs=32000, output="sos")

# 1. Polyphase Resampler
def resample_poly_32k(wav, orig_sr=48000, target_sr=32000):
    if orig_sr == target_sr:
        return wav.astype(np.float32)
    g = gcd(orig_sr, target_sr)
    return resample_poly(wav, target_sr // g, orig_sr // g).astype(np.float32)

# 2. High-Pass 100 Hz (32 kHz)
def butter_highpass_100hz(wav):
    return sosfilt(SOS_HP_100, wav).astype(np.float32)

# 3. Linear Peak Normalization (near-silent tidak diperkuat)
def linear_peak_norm(wav, target_peak=0.95):
    peak = float(np.max(np.abs(wav)))
    if peak > SILENT_EPS:
        return (wav / peak * target_peak).astype(np.float32)
    return wav.astype(np.float32)

# 4. Inline VAD Window Indices (500-8000 Hz, 32 kHz) - O(N) via cumsum
def _compute_vad_window_indices(wav, sr=32000, duration_sec=10.0):
    target_len = int(sr * duration_sec)
    cur_len = len(wav)
    if cur_len <= target_len:
        return 0, target_len
    bp = sosfilt(SOS_BP_VAD, wav).astype(np.float64)
    stride = int(sr * 0.25)
    n_steps = max(1, (cur_len - target_len) // stride + 1)
    cs = np.concatenate(([0.0], np.cumsum(bp ** 2)))
    starts = np.arange(n_steps) * stride
    energies = cs[starts + target_len] - cs[starts]
    best_start = int(starts[int(np.argmax(energies))])
    return best_start, best_start + target_len

# 5. Perch: 3 window 5.0s @ 32 kHz
def extract_perch_3windows_5s(wav, sr=32000):
    win_len = int(sr * 5.0)
    target_10s = int(sr * 10.0)
    cur_len = len(wav)

    if cur_len < win_len:
        pad_len = win_len - cur_len
        w_pad = np.pad(wav, (0, pad_len), mode="reflect" if pad_len < cur_len else "constant")
        w_pad = linear_peak_norm(w_pad, 0.95)
        return np.stack([w_pad, w_pad, w_pad], axis=0).astype(np.float32)
    elif cur_len < target_10s:
        t0, t2 = 0, cur_len - win_len
        t1 = (t0 + t2) // 2
        ws = [linear_peak_norm(wav[t : t + win_len], 0.95) for t in (t0, t1, t2)]
        return np.stack(ws, axis=0).astype(np.float32)
    else:
        t_start, t_end = _compute_vad_window_indices(wav, sr=sr, duration_sec=10.0)
        chunk10 = wav[t_start:t_end]
        stride = int(sr * 2.5)
        ws = [linear_peak_norm(chunk10[k * stride : k * stride + win_len], 0.95) for k in range(3)]
        return np.stack(ws, axis=0).astype(np.float32)

# 6. BirdNET: 3 window 3.0s @ 48 kHz
def extract_birdnet_3windows_3s(wav, sr=48000):
    chunk_len = int(sr * 3.0)
    cur_len = len(wav)
    if cur_len < chunk_len:
        wav = np.pad(wav, (0, chunk_len - cur_len), mode="wrap")
        cur_len = len(wav)
    w1 = wav[:chunk_len]
    mid = (cur_len - chunk_len) // 2
    w2 = wav[mid : mid + chunk_len]
    w3 = wav[-chunk_len:]
    return np.stack([w1, w2, w3], axis=0).astype(np.float32)

# 7. Dual-Stream Process Single Audio (single disk read)
def process_single_audio_dual(filepath):
    data, orig_sr = sf.read(str(filepath), dtype="float32")
    mono = data.mean(axis=1) if data.ndim > 1 else data
    dur_sec = len(mono) / orig_sr
    if mono.size == 0:   # guard file kosong
        mono = np.zeros(int(orig_sr * 0.1), dtype=np.float32)

    wav_32k = resample_poly_32k(mono, orig_sr=orig_sr, target_sr=32000)
    wav_hp_32k = butter_highpass_100hz(wav_32k)
    wins_32k = extract_perch_3windows_5s(wav_hp_32k, sr=32000)

    if orig_sr == 48000:
        wav_48k = mono
    else:
        g = gcd(orig_sr, 48000)
        wav_48k = resample_poly(mono, 48000 // g, orig_sr // g).astype(np.float32)
    wins_48k = extract_birdnet_3windows_3s(wav_48k, sr=48000)

    n_silent = int(((np.abs(wins_32k).max(axis=1) < SILENT_EPS) |
                    (np.abs(wins_48k).max(axis=1) < SILENT_EPS)).sum())
    return wins_32k, wins_48k, dur_sec, n_silent

# 8. Cache satu split ke array pre-allocated
def cache_split(filepaths, name):
    n = len(filepaths)
    arr_32k = np.empty((n, 3, 160000), dtype=np.float32)
    arr_48k = np.empty((n, 3, 144000), dtype=np.float32)
    durs = np.empty(n, dtype=np.float32)
    silent = np.zeros(n, dtype=np.int16)

    gen = Parallel(n_jobs=DSP_JOBS, backend="loky", return_as="generator")(
        delayed(process_single_audio_dual)(fp) for fp in filepaths
    )  # berurutan; butuh joblib >= 1.3
    for i, (w32, w48, d, s) in enumerate(gen):
        arr_32k[i] = w32
        arr_48k[i] = w48
        durs[i] = d
        silent[i] = s
    print(f"{name}_cached | n={n} | ram={(arr_32k.nbytes + arr_48k.nbytes) / 1024**3:.2f}GB")
    return arr_32k, arr_48k, durs, silent

print(f"pre_caching_dual_audio_to_ram (jobs={DSP_JOBS})...")

train_32k, train_48k, train_durs, train_silent = cache_split(train_df["filepath"].tolist(), "train")
test_32k, test_48k, test_durs, test_silent = cache_split(sub_df["filepath"].tolist(), "test")

n_tr, n_te = len(train_df), len(sub_df)
if not (train_32k.shape == (n_tr, 3, 160000) and test_32k.shape == (n_te, 3, 160000)
        and train_48k.shape == (n_tr, 3, 144000) and test_48k.shape == (n_te, 3, 144000)):
    raise ValueError("Shape array cache tidak sesuai")
for arr_name, arr in [("train_32k", train_32k), ("test_32k", test_32k),
                      ("train_48k", train_48k), ("test_48k", test_48k)]:
    if not np.isfinite(arr).all():
        raise ValueError(f"NaN/Inf terdeteksi di {arr_name}")

all_durs = np.concatenate([train_durs, test_durs])
short_5s = int((all_durs < 5.0).sum())
short_3s = int((all_durs < 3.0).sum())
silent_files = int((train_silent > 0).sum() + (test_silent > 0).sum())

elapsed_c3 = time.time() - t0_dsp
print(
    f"ram_caching_complete | train={len(train_32k)} | test={len(test_32k)} | "
    f"padded_perch(<5s)={short_5s} | wrapped_birdnet(<3s)={short_3s} | "
    f"silent_files={silent_files} | dur_min={all_durs.min():.2f}s | elapsed={elapsed_c3:.1f}s"
)

pre_caching_dual_audio_to_ram (jobs=4)...
train_cached | n=1826 | ram=6.20GB
test_cached | n=814 | ram=2.77GB
ram_caching_complete | train=1826 | test=814 | padded_perch(<5s)=0 | wrapped_birdnet(<3s)=0 | silent_files=0 | dur_min=5.27s | elapsed=94.2s


In [5]:
# Cell 4 - Batch Extraction Google Perch v2 (Embedding 1536-d + Candidate Logits 14-d) (revised)
import contextlib
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

t0_perch = time.time()

@contextlib.contextmanager
def silence_native_stderr():
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(devnull)

# 1. Muat Model Perch v2
perch_candidates = [
    Path("/kaggle/input/bird-vocalization-classifier/tensorFlow2/perch_v2/2"),
    Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2"),
    Path("/kaggle/input/google-perch-v2"),
    Path("/kaggle/input/perch-v2"),
    WORK_DIR / "perch_v2",
    Path("perch_v2"),
]

perch_model = None
resolved_model_dir = None
for p_dir in perch_candidates:
    if p_dir.exists():
        try:
            perch_model = hub.load(str(p_dir))
            resolved_model_dir = p_dir.resolve()
            print(f"perch_model_loaded | type=local | path={resolved_model_dir}")
            break
        except Exception as e:
            print(f"  [warn] Gagal memuat dari {p_dir}: {e}")

if perch_model is None:
    tf_url = "https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2"
    print(f"loading_perch_from_tfhub: {tf_url}...")
    try:
        resolved_path_str = hub.resolve(tf_url)
        resolved_model_dir = Path(resolved_path_str)
        perch_model = hub.load(resolved_path_str)
    except Exception as e:
        raise RuntimeError(
            "Model Perch v2 tidak ditemukan secara lokal dan gagal diunduh. "
            "Tambahkan model lewat 'Add Input' di Kaggle atau aktifkan Internet. "
            f"Detail: {e}"
        )
    print(f"perch_model_loaded | type=tfhub | cache={resolved_model_dir}")

infer_fn = perch_model.signatures["serving_default"]

# 2. Verifikasi signature
with silence_native_stderr():
    dummy_out = infer_fn(inputs=tf.constant(np.zeros((1, 160000), dtype=np.float32)))
logit_key = next((k for k in ["label", "logits", "output_0", "predictions"] if k in dummy_out), None)
if logit_key is None:
    logit_key = [k for k in dummy_out.keys() if k not in ("embedding", "spatial_embedding", "spectrogram")][0]

emb_dim = int(dummy_out["embedding"].shape[-1])
num_taxa = int(dummy_out[logit_key].shape[-1])
print(f"perch_signature_verified | emb_dim={emb_dim} | num_taxa={num_taxa} | logit_key='{logit_key}'")
check(emb_dim == 1536 and num_taxa == 14795, f"Dimensi model tidak sesuai: emb={emb_dim}, taxa={num_taxa}")

# 3. Resolusi labels.csv
label_candidates = []
if resolved_model_dir and resolved_model_dir.exists():
    label_candidates.extend(sorted(resolved_model_dir.rglob("*.csv")))
if WORK_DIR.exists():
    label_candidates.extend(sorted(WORK_DIR.rglob("*label*.csv")))
for p_in in perch_candidates:
    if p_in.exists() and (resolved_model_dir is None or p_in.resolve() != resolved_model_dir):
        label_candidates.extend(sorted(p_in.rglob("*.csv")))

labels_df = None
labels_path = None
for p_lbl in label_candidates:
    try:
        df_c = pd.read_csv(p_lbl)
        if len(df_c) == num_taxa:
            labels_df, labels_path = df_c, p_lbl
            break
        df_c_nohead = pd.read_csv(p_lbl, header=None)
        if len(df_c_nohead) == num_taxa:
            labels_df, labels_path = df_c_nohead, p_lbl
            labels_df.columns = [f"col_{i}" for i in range(labels_df.shape[1])]
            break
    except Exception:
        continue

check(labels_df is not None and len(labels_df) == num_taxa, "labels.csv (14.795 baris) tidak ditemukan")
print(f"labels_loaded | path={labels_path}")

# 4. Indeks taksa target
BONDOL_SCI = "Lonchura leucogastroides"
KUTILANG_SCI = "Pycnonotus aurigaster"
CENDRA_SCI = [
    "Cicinnurus regius", "Paradisaea apoda", "Paradisaea decora", "Paradisaea guilielmi",
    "Paradisaea minor", "Paradisaea raggiana", "Paradisaea rubra", "Seleucidis melanoleucus",
]

cols_to_check = [c for c in labels_df.columns if not pd.api.types.is_numeric_dtype(labels_df[c])]
check(bool(cols_to_check), "Tidak ada kolom teks di labels.csv")

def find_taxa(sci_name):
    matched = set()
    target = sci_name.strip().lower()
    for c in cols_to_check:
        mask = labels_df[c].astype(str).str.strip().str.lower() == target
        matched.update(labels_df.index[mask].tolist())
    return sorted(i for i in matched if 0 <= i < num_taxa)

def label_text(i):
    return " | ".join(str(v) for v in labels_df.iloc[i].tolist())

def resolve_single(sci_name):
    hits = find_taxa(sci_name)
    check(len(hits) == 1, f"'{sci_name}': diharapkan tepat 1 taksa, diperoleh {len(hits)}: {[label_text(i) for i in hits]}")
    return hits[0]

idx_bondol = resolve_single(BONDOL_SCI)
idx_kutilang = resolve_single(KUTILANG_SCI)
cendra_candidates = [resolve_single(sp) for sp in CENDRA_SCI]
check(len(set(cendra_candidates + [idx_bondol, idx_kutilang])) == 2 + len(CENDRA_SCI), "Ada indeks taksa duplikat")

print(f"  Bondol    -> [{idx_bondol}] {label_text(idx_bondol)}")
print(f"  Kutilang  -> [{idx_kutilang}] {label_text(idx_kutilang)}")
for i in cendra_candidates:
    print(f"  Cendra    -> [{i}] {label_text(i)}")

all_target_cands = [idx_bondol, idx_kutilang] + list(cendra_candidates)
non_target_mask = np.ones(num_taxa, dtype=bool)
non_target_mask[all_target_cands] = False
n_cand_cendra = len(cendra_candidates)
n_summary_dim = 1 + 1 + n_cand_cendra + 4
check(n_cand_cendra == 8 and n_summary_dim == 14,
      f"Ekspektasi 8 kandidat Cendrawasih dan 14 dimensi, diperoleh {n_cand_cendra} dan {n_summary_dim}")

summary_layout = ["bondol", "kutilang"] + CENDRA_SCI + ["max_nt", "top5_nt", "energy", "entropy"]
with open(WORK_DIR / "perch_taxa_indices.json", "w") as f:
    json.dump({
        "bondol": int(idx_bondol),
        "kutilang": int(idx_kutilang),
        "cendra": {sp: int(i) for sp, i in zip(CENDRA_SCI, cendra_candidates)},
        "summary_layout": summary_layout,
        "summary_dim": int(n_summary_dim),
        "logit_key": logit_key,
        "model_dir": str(resolved_model_dir),
        "labels_path": str(labels_path),
    }, f, indent=2)

# 5. Ekstraksi batch dari RAM (32 kHz)
def extract_perch_features_from_ram(arr_32k, batch_size=32):
    n_files, n_windows, win_len = arr_32k.shape
    total_windows = n_files * n_windows
    flat_wavs = arr_32k.reshape(total_windows, win_len)

    embs_out = np.empty((total_windows, emb_dim), dtype=np.float32)
    summs_out = np.empty((total_windows, n_summary_dim), dtype=np.float32)

    for i in range(0, total_windows, batch_size):
        batch = flat_wavs[i : i + batch_size]
        res = infer_fn(inputs=tf.constant(batch))

        e_b = res["embedding"].numpy()
        l_b = res[logit_key].numpy()
        if e_b.ndim > 2:
            e_b = np.mean(e_b, axis=1)
        if l_b.ndim > 2:
            l_b = np.mean(l_b, axis=1)

        e_b = e_b / (np.linalg.norm(e_b, axis=-1, keepdims=True) + 1e-12)

        z_b = l_b[:, idx_bondol, None]
        z_k = l_b[:, idx_kutilang, None]
        z_c = l_b[:, cendra_candidates]
        nt_logits = l_b[:, non_target_mask]
        max_nt = np.max(nt_logits, axis=1, keepdims=True)
        top5_nt = np.mean(np.partition(nt_logits, -5, axis=1)[:, -5:], axis=1, keepdims=True)

        l64 = l_b.astype(np.float64)
        max_l = np.max(l64, axis=1, keepdims=True)
        exp_l = np.exp(l64 - max_l)
        sum_exp = np.sum(exp_l, axis=1, keepdims=True)
        probs = exp_l / sum_exp
        entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1, keepdims=True)
        energy = max_l + np.log(sum_exp)

        s_b = np.hstack([z_b, z_k, z_c, max_nt, top5_nt, energy, entropy]).astype(np.float32)

        embs_out[i : i + len(batch)] = e_b
        summs_out[i : i + len(batch)] = s_b

        if (i // batch_size) % 32 == 0:
            print(f"  {i + len(batch)}/{total_windows} windows", flush=True)

    return (embs_out.reshape(n_files, n_windows, emb_dim),
            summs_out.reshape(n_files, n_windows, n_summary_dim))

print("extracting_perch_features_train...")
with silence_native_stderr():
    train_embs_3w, train_cand_logits_3w = extract_perch_features_from_ram(train_32k, batch_size=32)

print("extracting_perch_features_test...")
with silence_native_stderr():
    test_embs_3w, test_cand_logits_3w = extract_perch_features_from_ram(test_32k, batch_size=32)

# 6. Validasi, lalu simpan
n_tr, n_te = len(train_df), len(sub_df)
check(train_embs_3w.shape == (n_tr, 3, 1536) and test_embs_3w.shape == (n_te, 3, 1536), "Shape embedding tidak sesuai")
check(train_cand_logits_3w.shape == (n_tr, 3, n_summary_dim) and test_cand_logits_3w.shape == (n_te, 3, n_summary_dim),
      "Shape ringkasan logit tidak sesuai")
for arr_name, arr in [("train_embs", train_embs_3w), ("test_embs", test_embs_3w),
                      ("train_cand", train_cand_logits_3w), ("test_cand", test_cand_logits_3w)]:
    check(bool(np.isfinite(arr).all()), f"NaN/Inf terdeteksi di {arr_name}")

# 7. Sanity check: rata-rata logit target per label (hanya peringatan)
cs = train_cand_logits_3w.mean(axis=1)
sanity = pd.DataFrame({
    "label": train_df["label"].values,
    "z_bondol": cs[:, 0],
    "z_kutilang": cs[:, 1],
    "z_cendra_max": cs[:, 2:2 + n_cand_cendra].max(axis=1),
}).groupby("label").mean().round(2)
print(sanity)
for lbl, col in [("Bondol Jawa", "z_bondol"), ("Kutilang", "z_kutilang"), ("Cendrawasih", "z_cendra_max")]:
    if sanity[col].idxmax() != lbl:
        print(f"  [warn] {col} tertinggi di '{sanity[col].idxmax()}', bukan '{lbl}'. Cek indeks taksa.")

np.save(WORK_DIR / "perch_v2_train_emb.npy", train_embs_3w)
np.save(WORK_DIR / "train_raw_cand_summary.npy", train_cand_logits_3w)
np.save(WORK_DIR / "perch_v2_test_emb.npy", test_embs_3w)
np.save(WORK_DIR / "test_raw_cand_summary.npy", test_cand_logits_3w)

elapsed_c4 = time.time() - t0_perch
print(
    f"perch_extracted | train_emb={train_embs_3w.shape} | test_emb={test_embs_3w.shape} | "
    f"candidate_logits={train_cand_logits_3w.shape[-1]} | elapsed={elapsed_c4:.1f}s"
)

loading_perch_from_tfhub: https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2...


I0000 00:00:1792433037.872902      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792433037.879095      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


perch_model_loaded | type=tfhub | cache=/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
perch_signature_verified | emb_dim=1536 | num_taxa=14795 | logit_key='label'
labels_loaded | path=/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2/assets/labels.csv
  Bondol    -> [7329] Lonchura leucogastroides
  Kutilang  -> [11701] Pycnonotus aurigaster
  Cendra    -> [2893] Cicinnurus regius
  Cendra    -> [9634] Paradisaea apoda
  Cendra    -> [9635] Paradisaea decora
  Cendra    -> [9636] Paradisaea guilielmi
  Cendra    -> [9637] Paradisaea minor
  Cendra    -> [9638] Paradisaea raggiana
  Cendra    -> [9639] Paradisaea rubra
  Cendra    -> [12538] Seleucidis melanoleucus
extracting_perch_features_train...
  32/5478 windows
  1056/5478 windows
  2080/5478 windows
  3104/5478 windows
  4128/5478 windows
  5152/5478 windows
extracting_perch_features_test...
  32/2442 windows
  1056/2442 windows
  2080/2442 windows
             z_bon

In [6]:
# Cell 5 - Batch Extraction Official BirdNET v2.4 (Logit 4-d Summary) (revised)
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

t0_bnet = time.time()

# 1. Pustaka resmi birdnet & load model (v2.4)
try:
    import birdnet
except ImportError:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "birdnet"], check=True)
        import birdnet
    except Exception as e:
        raise RuntimeError(f"Gagal install 'birdnet' (cek Internet/pip di Kaggle). Detail: {e}")

try:
    bnet_model = birdnet.load("acoustic", "2.4", "tf", library="litert")
except Exception as e:
    print(f"litert fallback to tf: {e}")
    bnet_model = birdnet.load("acoustic", "2.4", "tf")

species_list = list(bnet_model.species_list)
N_SP = len(species_list)
SR_BNET = 48000
BNET_VER = getattr(birdnet, "__version__", "unknown")

# 2. Resolusi taksa target (nama ilmiah = bagian sebelum "_"; ambil 2 kata pertama)
def _sci(s):
    parts = s.split("_")[0].replace("_", " ").split()
    return " ".join(parts[:2]) if len(parts) >= 2 else s

sci_of = [_sci(s) for s in species_list]
genus_of = [n.split(" ")[0] for n in sci_of]
sci_lower = [n.lower() for n in sci_of]

def find_species(sci_name):
    hits = [i for i, n in enumerate(sci_lower) if n == sci_name.lower()]
    check(len(hits) == 1, f"'{sci_name}': diharapkan 1 taksa di BirdNET, diperoleh {len(hits)}")
    return hits[0]

IDX_BONDOL = find_species("Lonchura leucogastroides")
IDX_KUTILANG = find_species("Pycnonotus aurigaster")

PARADISAEIDAE_GENERA = {
    "Astrapia", "Cicinnurus", "Drepanornis", "Epimachus", "Loboparadisea", "Lophorina",
    "Lycocorax", "Macgregoria", "Manucodia", "Paradigalla", "Paradisaea", "Parotia",
    "Phonygammus", "Pteridophora", "Ptiloris", "Seleucidis", "Semioptera",
}
CENDRA_IDX = [i for i, g in enumerate(genus_of) if g in PARADISAEIDAE_GENERA]
check(len(CENDRA_IDX) > 0, "Tidak ada taksa Paradisaeidae ditemukan di BirdNET")

print(f"birdnet_taxa_resolved | total={N_SP} | Bondol={IDX_BONDOL} | Kutilang={IDX_KUTILANG} | "
      f"Cendra={len(CENDRA_IDX)} | birdnet={BNET_VER}")
for i in CENDRA_IDX:
    print(f"  Cendra -> [{i}] {species_list[i]}")

# Info: spesies Cendrawasih Perch (Cell 4) yang tidak ada di BirdNET
if "CENDRA_SCI" in globals():
    miss = [s for s in CENDRA_SCI if s.lower() not in set(sci_lower)]
    if miss:
        print(f"  [info] Ada di Perch tapi tidak di BirdNET: {miss}")

NONTARGET = np.ones(N_SP, dtype=bool)
NONTARGET[[IDX_BONDOL, IDX_KUTILANG] + CENDRA_IDX] = False

# 3. Pemetaan matriks logit penuh; mengembalikan juga jumlah spesies tanpa skor
def to_full_matrix(res, n_inputs):
    full = np.full((n_inputs, N_SP), fill_value=-50.0, dtype=np.float32)
    ids = np.asarray(res.species_ids)[:, 0, :]
    vals = np.asarray(res.species_probs)[:, 0, :]  # logit (apply_sigmoid=False)
    max_missing = 0
    for r in range(n_inputs):
        ok = (ids[r] >= 0) & (ids[r] < N_SP)
        full[r, ids[r][ok]] = vals[r][ok]
        max_missing = max(max_missing, N_SP - int(ok.sum()))
    return full, max_missing

# 4. Ekstraksi batch + checkpoint beridentitas
CHUNK_FILES = 64
CKPT_DIR = WORK_DIR / "birdnet_ckpt"
CKPT_DIR.mkdir(exist_ok=True)

cfg_sig = hashlib.md5(json.dumps(
    {"n_sp": N_SP, "b": IDX_BONDOL, "k": IDX_KUTILANG, "c": CENDRA_IDX,
     "chunk": CHUNK_FILES, "ver": BNET_VER}
).encode()).hexdigest()[:8]

if "silence_native_stderr" not in globals():
    import contextlib
    silence_native_stderr = contextlib.nullcontext

MISSING_MAX = 0

def extract_birdnet_split_from_ram(arr_48k, tag):
    global MISSING_MAX
    n_files = len(arr_48k)
    out = np.full((n_files, 3, 4), np.nan, dtype=np.float32)
    data_sig = hashlib.md5(np.ascontiguousarray(arr_48k[:, :, ::997]).tobytes()).hexdigest()[:8]

    for s in tqdm(range(0, n_files, CHUNK_FILES), desc=f"BirdNET {tag}"):
        chunk_len = min(CHUNK_FILES, n_files - s)
        ck = CKPT_DIR / f"{tag}_{cfg_sig}_{data_sig}_{s:06d}.npy"

        if ck.exists():
            try:
                blk = np.load(ck)
                if blk.shape == (chunk_len, 3, 4) and np.isfinite(blk).all():
                    out[s : s + chunk_len] = blk
                    continue
            except Exception:
                pass  # checkpoint rusak, hitung ulang

        inputs = [(arr_48k[i, w], SR_BNET) for i in range(s, s + chunk_len) for w in range(3)]

        with silence_native_stderr():
            res = bnet_model.predict_arrays(
                inputs, top_k=N_SP, apply_sigmoid=False, default_confidence_threshold=-1e9
            )
        full, miss = to_full_matrix(res, len(inputs))
        MISSING_MAX = max(MISSING_MAX, miss)
        full = full.reshape(chunk_len, 3, N_SP)

        blk = np.stack(
            [
                full[:, :, IDX_BONDOL],
                full[:, :, IDX_KUTILANG],
                full[:, :, CENDRA_IDX].max(axis=-1),
                full[:, :, NONTARGET].max(axis=-1),
            ],
            axis=-1,
        ).astype(np.float32)

        tmp = ck.with_name(ck.stem + ".tmp.npy")
        np.save(tmp, blk)
        os.replace(tmp, ck)  # atomik
        out[s : s + chunk_len] = blk

    return out

print("extracting_birdnet_train...")
train_birdnet_sum = extract_birdnet_split_from_ram(train_48k, "train")
print("extracting_birdnet_test...")
test_birdnet_sum = extract_birdnet_split_from_ram(test_48k, "test")

# 5. Validasi, lalu simpan
n_tr, n_te = len(train_df), len(sub_df)
check(train_birdnet_sum.shape == (n_tr, 3, 4) and test_birdnet_sum.shape == (n_te, 3, 4),
      "Shape ringkasan BirdNET tidak sesuai")
check(bool(np.isfinite(train_birdnet_sum).all() and np.isfinite(test_birdnet_sum).all()),
      "NaN/Inf di ringkasan BirdNET")
if MISSING_MAX > 0:
    print(f"  [warn] Ada input dengan {MISSING_MAX} spesies tanpa skor (terisi -50.0)")

np.save(WORK_DIR / "birdnet_train_summary.npy", train_birdnet_sum)
np.save(WORK_DIR / "birdnet_test_summary.npy", test_birdnet_sum)

# 6. Sanity check: rata-rata fitur per kelas [Bondol, Kutilang, Cendra_max, NT_max]
print("sanity_mean_per_class (kolom: Bondol | Kutilang | Cendra_max | NT_max)")
means = {}
for lab in sorted(train_df["label"].unique()):
    means[lab] = train_birdnet_sum[(train_df["label"] == lab).values].mean(axis=(0, 1))
    print(f"  {lab:<12} {np.round(means[lab], 2)}")
for lab, col in [("Bondol Jawa", 0), ("Kutilang", 1), ("Cendrawasih", 2)]:
    best = max(means, key=lambda k: means[k][col])
    if best != lab:
        print(f"  [warn] kolom {col} tertinggi di '{best}', bukan '{lab}'. Cek indeks taksa BirdNET.")

elapsed_c5 = time.time() - t0_bnet
print(
    f"birdnet_extracted | train_sum={train_birdnet_sum.shape} | "
    f"test_sum={test_birdnet_sum.shape} | nan=0 | elapsed={elapsed_c5:.1f}s"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.7/208.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.4 MB/s eta 0:00:00


birdnet_taxa_resolved | total=6522 | Bondol=3260 | Kutilang=5133 | Cendra=17 | birdnet=unknown
  Cendra -> [1312] Cicinnurus magnificus_Magnificent Bird-of-Paradise
  Cendra -> [1313] Cicinnurus regius_King Bird-of-Paradise
  Cendra -> [1314] Cicinnurus respublica_Wilson's Bird-of-Paradise
  Cendra -> [1955] Drepanornis albertisi_Black-billed Sicklebill
  Cendra -> [2157] Epimachus fastosus_Black Sicklebill
  Cendra -> [2158] Epimachus meyeri_Brown Sicklebill
  Cendra -> [3277] Lophorina superba_Greater Lophorina
  Cendra -> [3314] Lycocorax pyrrhopterus_Paradise-crow
  Cendra -> [4218] Paradisaea minor_Lesser Bird-of-Paradise
  Cendra -> [4219] Paradisaea rubra_Red Bird-of-Paradise
  Cendra -> [4485] Phonygammus keraudrenii_Trumpet Manucode
  Cendra -> [5029] Pteridophora alberti_King-of-Saxony Bird-of-Paradise
  Cendra -> [5109] Ptiloris magnificus_Magnificent Riflebird
  Cendra -> [5110] Ptiloris paradiseus_Paradise Riflebird
  Cendra -> [5111] Ptiloris victoriae_Victoria's Riflebir

BirdNET train:   0%|          | 0/29 [00:00<?, ?it/s]

extracting_birdnet_test...


BirdNET test:   0%|          | 0/13 [00:00<?, ?it/s]

sanity_mean_per_class (kolom: Bondol | Kutilang | Cendra_max | NT_max)
  Bondol Jawa  [-14.07 -11.72 -10.24  -1.75]
  Cendrawasih  [-14.08 -11.6   -6.42  -2.14]
  Kutilang     [-15.2   -3.92 -10.19  -1.57]
birdnet_extracted | train_sum=(1826, 3, 4) | test_sum=(814, 3, 4) | nan=0 | elapsed=563.0s


In [7]:
# Cell 6 - Explicit Model Deletion, VRAM Purge, & Memory Sanity Check (revised)
import gc
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
import torch
import tensorflow as tf

# 1. Hapus objek model berat & buffer audio mentah dari RAM
heavy_vars = [
    "perch_model", "infer_fn", "bnet_model", "dummy_out", "flat_wavs",
    "train_32k", "test_32k", "train_48k", "test_48k",
]
freed = []
for var_name in heavy_vars:
    if var_name in globals():
        del globals()[var_name]
        freed.append(var_name)

# 2. Matikan worker loky (Cell 3) yang masih menahan RAM
try:
    from joblib.externals.loky import get_reusable_executor
    get_reusable_executor().shutdown(wait=True)
except Exception as e:
    print(f"loky_shutdown_skipped: {e}")

# 3. Bersihkan session backend & VRAM (memori GPU milik TF tidak dikembalikan ke OS)
tf.keras.backend.clear_session()
gc.collect()
purge_vram()

# 4. Verifikasi artefak fitur di disk
n_tr, n_te = len(train_df), len(sub_df)
with open(WORK_DIR / "perch_taxa_indices.json") as f:
    summary_dim = int(json.load(f)["summary_dim"])

expected_artifacts = [
    ("perch_v2_train_emb.npy", (n_tr, 3, 1536)),
    ("perch_v2_test_emb.npy", (n_te, 3, 1536)),
    ("train_raw_cand_summary.npy", (n_tr, 3, summary_dim)),
    ("test_raw_cand_summary.npy", (n_te, 3, summary_dim)),
    ("birdnet_train_summary.npy", (n_tr, 3, 4)),
    ("birdnet_test_summary.npy", (n_te, 3, 4)),
]

for fname, exp_shape in expected_artifacts:
    fpath = WORK_DIR / fname
    check(fpath.exists(), f"Artefak wajib hilang: {fname}")
    arr_meta = np.load(fpath, mmap_mode="r")
    check(arr_meta.shape == exp_shape, f"Dimensi {fname} salah: {arr_meta.shape} vs {exp_shape}")
    check(bool(np.isfinite(arr_meta).all()), f"NaN/Inf terdeteksi di {fname}")
    del arr_meta

folds_path = WORK_DIR / "metadata_with_folds.csv"
check(folds_path.exists(), "Artefak wajib hilang: metadata_with_folds.csv")
saved_meta = pd.read_csv(folds_path, dtype={"file_id": str, "source_group": str})
check({"file_id", "fold", "source_group"}.issubset(saved_meta.columns),
      "Kolom file_id/fold/source_group tidak ada di metadata_with_folds.csv")
check(saved_meta["file_id"].tolist() == train_df["file_id"].tolist(),
      "Urutan file_id di metadata_with_folds.csv berbeda dari train_df (fitur akan tidak sejajar)")
check((saved_meta["fold"].values == train_df["fold"].values).all(), "Fold di disk berbeda dari train_df")
check((saved_meta["source_group"].values == train_df["source_group"].astype(str).values).all(),
      "source_group di disk berbeda dari train_df")

# 5. Status memori RAM & VRAM
ram_used_gb = psutil.Process(os.getpid()).memory_info().rss / (1024**3)
ram_sys_gb = psutil.virtual_memory().used / (1024**3)

vram_report = []
if torch.cuda.is_available():
    for d in range(torch.cuda.device_count()):
        free_b, total_b = torch.cuda.mem_get_info(d)
        vram_report.append(f"cuda:{d}={free_b / 1024**3:.2f}/{total_b / 1024**3:.2f}GB")
    free0 = torch.cuda.mem_get_info(0)[0] / 1024**3
    if free0 < 10.0:
        print(f"  [warn] VRAM bebas hanya {free0:.2f}GB (TF belum melepas memori). "
              f"Jika PyTorch OOM: restart kernel, lalu muat fitur dari .npy di WORK_DIR.")

print(
    f"memory_sanitized | freed={len(freed)} vars | artifacts_ok={len(expected_artifacts) + 1} | "
    f"proc_ram={ram_used_gb:.2f}GB | sys_ram={ram_sys_gb:.2f}GB | "
    f"vram_free/total: {' '.join(vram_report) if vram_report else 'n/a'} | "
    f"pytorch_cuda_ready={torch.cuda.is_available()}"
)

  [warn] VRAM bebas hanya 7.34GB (TF belum melepas memori). Jika PyTorch OOM: restart kernel, lalu muat fitur dari .npy di WORK_DIR.
memory_sanitized | freed=8 vars | artifacts_ok=7 | proc_ram=3.27GB | sys_ram=2.92GB | vram_free/total: cuda:0=7.34/14.56GB cuda:1=14.46/14.56GB | pytorch_cuda_ready=True


In [8]:
# Cell 7 - RESOLVE: Global Taxa Lock & Dual Feature Assembly (1543-d hybrid vs 1536-d pure) (revised)
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

t0_c7 = time.time()

RESOLVE_GLOBAL_CENDRA_COL = 0
EXPECTED_LOCK_SPECIES = "Cicinnurus regius"
EXPECTED_LOCK_IDX = 2893

# 1. Muat fitur dan konfigurasi taksa
with open(WORK_DIR / "perch_taxa_indices.json") as f:
    taxa_meta = json.load(f)

cendra_sp_names = list(taxa_meta["cendra"].keys())
n_cand = len(cendra_sp_names)
check(cendra_sp_names[RESOLVE_GLOBAL_CENDRA_COL] == EXPECTED_LOCK_SPECIES,
      f"Kolom {RESOLVE_GLOBAL_CENDRA_COL} bukan {EXPECTED_LOCK_SPECIES}: {cendra_sp_names[RESOLVE_GLOBAL_CENDRA_COL]}")
check(taxa_meta["cendra"][EXPECTED_LOCK_SPECIES] == EXPECTED_LOCK_IDX,
      f"Indeks taksa {EXPECTED_LOCK_SPECIES} = {taxa_meta['cendra'][EXPECTED_LOCK_SPECIES]}, bukan {EXPECTED_LOCK_IDX}")

train_raw_sum = np.load(WORK_DIR / "train_raw_cand_summary.npy")
test_raw_sum = np.load(WORK_DIR / "test_raw_cand_summary.npy")
X_tr_emb = np.load(WORK_DIR / "perch_v2_train_emb.npy")
X_te_emb = np.load(WORK_DIR / "perch_v2_test_emb.npy")

n_tr, n_te = len(train_df), len(sub_df)
check(train_raw_sum.shape[0] == n_tr and X_tr_emb.shape[0] == n_tr, "Baris fitur train tidak sejajar dengan train_df")
check(test_raw_sum.shape[0] == n_te and X_te_emb.shape[0] == n_te, "Baris fitur test tidak sejajar dengan sub_df")
check(train_raw_sum.shape[-1] == taxa_meta["summary_dim"], "summary_dim tidak sesuai")

COL_B, COL_K = 0, 1
cendra_cols = list(range(2, 2 + n_cand))
COL_MAX_NT = 2 + n_cand
COL_TOP5_NT = 2 + n_cand + 1
COL_ENERGY = 2 + n_cand + 2

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
mapped = train_df["label"].map(CLASS_TO_IDX)
check(mapped.notna().all(), f"Label tak dikenal: {sorted(train_df.loc[mapped.isna(), 'label'].unique())}")
train_df["target"] = mapped.astype(int)
y_target = train_df["target"].values
folds = train_df["fold"].values

# 2. Verifikasi global taxa lock (catatan: pemilihan Cicinnurus dibuat setelah melihat semua fold,
#    jadi angka held-out di bawah optimistis)
global_c_col = cendra_cols[RESOLVE_GLOBAL_CENDRA_COL]
y_cendra = (y_target == 1).astype(int)
cand_max = np.max(train_raw_sum[:, :, 2:2 + n_cand], axis=1)

def safe_auc(y_true, scores):
    return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else np.nan

print(f"RESOLVE taxa verification ({EXPECTED_LOCK_SPECIES} idx={EXPECTED_LOCK_IDX}, global lock):")
for fold_id in sorted(np.unique(folds)):
    tr_mask, va_mask = folds != fold_id, folds == fold_id
    auc_tr = safe_auc(y_cendra[tr_mask], cand_max[tr_mask, RESOLVE_GLOBAL_CENDRA_COL])
    auc_va = safe_auc(y_cendra[va_mask], cand_max[va_mask, RESOLVE_GLOBAL_CENDRA_COL])
    print(f"  Fold {int(fold_id)}: train AUROC={auc_tr:.4f} | held-out AUROC={auc_va:.4f} "
          f"(n_pos_val={int(y_cendra[va_mask].sum())})")

print("\nDay 4 comparison (fold-dependent selection):")
for fold_id in sorted(np.unique(folds)):
    tr_mask = folds != fold_id
    aucs = [safe_auc(y_cendra[tr_mask], cand_max[tr_mask, j]) for j in range(n_cand)]
    best_j = int(np.nanargmax(aucs))
    va_auc_best = safe_auc(y_cendra[folds == fold_id], cand_max[folds == fold_id, best_j])
    va_auc_global = safe_auc(y_cendra[folds == fold_id], cand_max[folds == fold_id, RESOLVE_GLOBAL_CENDRA_COL])
    print(f"  Fold {int(fold_id)}: Day4='{cendra_sp_names[best_j]}' (held={va_auc_best:.4f}) | "
          f"Global {EXPECTED_LOCK_SPECIES} (held={va_auc_global:.4f}) | delta={va_auc_global - va_auc_best:+.4f}")

oof_auc = safe_auc(y_cendra, cand_max[:, RESOLVE_GLOBAL_CENDRA_COL])
print(f"\n  Pooled AUROC (semua train, {EXPECTED_LOCK_SPECIES}): {oof_auc:.4f}")

# 3. Ringkasan 7-d dengan taksa global
def build_summary_7d(raw_sum, c_col):
    z_b = raw_sum[:, :, COL_B:COL_B + 1]
    z_k = raw_sum[:, :, COL_K:COL_K + 1]
    z_c = raw_sum[:, :, c_col:c_col + 1]
    max_nt = raw_sum[:, :, COL_MAX_NT:COL_MAX_NT + 1]
    top5_nt = raw_sum[:, :, COL_TOP5_NT:COL_TOP5_NT + 1]
    energy = raw_sum[:, :, COL_ENERGY:COL_ENERGY + 1]
    margin_ood = max_nt - np.maximum(np.maximum(z_b, z_k), z_c)
    return np.concatenate([z_b, z_c, z_k, max_nt, top5_nt, margin_ood, energy], axis=2).astype(np.float32)

def fit_zscore(s7):
    mu = s7.mean(axis=(0, 1), keepdims=True)
    sd = s7.std(axis=(0, 1), keepdims=True) + 1e-6
    return mu, sd

s7_train_global = build_summary_7d(train_raw_sum, global_c_col)
mu_global, sd_global = fit_zscore(s7_train_global)   # fit di seluruh train (tanpa label, bocor kecil untuk CV)
np.savez(WORK_DIR / "resolve_zscore_stats.npz", mu=mu_global, sd=sd_global)

train_hybrid_1543 = np.concatenate([X_tr_emb, (s7_train_global - mu_global) / sd_global], axis=-1).astype(np.float32)
test_hybrid_1543 = np.concatenate(
    [X_te_emb, (build_summary_7d(test_raw_sum, global_c_col) - mu_global) / sd_global], axis=-1
).astype(np.float32)

check(train_hybrid_1543.shape == (n_tr, 3, 1543) and bool(np.isfinite(train_hybrid_1543).all()), "Hybrid train tidak valid")
check(test_hybrid_1543.shape == (n_te, 3, 1543) and bool(np.isfinite(test_hybrid_1543).all()), "Hybrid test tidak valid")

np.save(WORK_DIR / "train_hybrid_1543.npy", train_hybrid_1543)
np.save(WORK_DIR / "test_hybrid_1543.npy", test_hybrid_1543)

emb_std = float(X_tr_emb.std())
sum_std = float(((s7_train_global - mu_global) / sd_global).std())
print(f"\nfeature_variants_ready:")
print(f"  Variant A (hybrid 1543-d) : train={train_hybrid_1543.shape} | test={test_hybrid_1543.shape}")
print(f"  Variant B (pure 1536-d)   : train={X_tr_emb.shape} | test={X_te_emb.shape}")
print(f"  Skala: std embedding={emb_std:.4f} vs std summary(z)={sum_std:.4f} -> pertimbangkan scaling di model")
print(f"  Global taxa locked: '{cendra_sp_names[RESOLVE_GLOBAL_CENDRA_COL]}' (col={global_c_col})")

meta_c7 = {
    "resolve_strategy": "global_lock",
    "locked_species": cendra_sp_names[RESOLVE_GLOBAL_CENDRA_COL],
    "locked_idx": int(taxa_meta["cendra"][EXPECTED_LOCK_SPECIES]),
    "locked_col": int(global_c_col),
    "zscore_stats": "resolve_zscore_stats.npz",
}
with open(WORK_DIR / "resolve_taxa_meta.json", "w") as fh:
    json.dump(meta_c7, fh, indent=2)

print(f"cell7_completed | elapsed={time.time() - t0_c7:.1f}s")

RESOLVE taxa verification (Cicinnurus regius idx=2893, global lock):
  Fold 0: train AUROC=0.8911 | held-out AUROC=0.8897 (n_pos_val=35)
  Fold 1: train AUROC=0.8501 | held-out AUROC=0.9721 (n_pos_val=29)
  Fold 2: train AUROC=0.9280 | held-out AUROC=0.7458 (n_pos_val=16)
  Fold 3: train AUROC=0.8897 | held-out AUROC=0.9235 (n_pos_val=3)

Day 4 comparison (fold-dependent selection):
  Fold 0: Day4='Cicinnurus regius' (held=0.8897) | Global Cicinnurus regius (held=0.8897) | delta=+0.0000
  Fold 1: Day4='Cicinnurus regius' (held=0.9721) | Global Cicinnurus regius (held=0.9721) | delta=+0.0000
  Fold 2: Day4='Paradisaea rubra' (held=0.7386) | Global Cicinnurus regius (held=0.7458) | delta=+0.0072
  Fold 3: Day4='Paradisaea rubra' (held=0.5757) | Global Cicinnurus regius (held=0.9235) | delta=+0.3479

  Pooled AUROC (semua train, Cicinnurus regius): 0.8917

feature_variants_ready:
  Variant A (hybrid 1543-d) : train=(1826, 3, 1543) | test=(814, 3, 1543)
  Variant B (pure 1536-d)   : train=

In [9]:
# Cell 8 - A/B Test: PerchBottleneckCosineHead on hybrid 1543-d vs pure 1536-d (revised)
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

t0_c8 = time.time()

SEED = 42
N_EPOCHS_HEAD = 40
BATCH_SIZE_HEAD = 32
EMB_DIM = 256
NUM_CLASSES = 3

check("X_tr_emb" in globals() and "X_te_emb" in globals(), "X_tr_emb/X_te_emb belum ada (jalankan Cell 7)")
y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)
n_tr, n_te = len(train_df), len(sub_df)

class CosineLinear(nn.Module):
    def __init__(self, in_features, num_classes=3, init_tau=16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype=torch.float32))
    def forward(self, x):
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(self.weight, p=2, dim=-1)
        return torch.clamp(self.tau, min=1.0, max=32.0) * F.linear(x_norm, w_norm)

class PerchBottleneckCosineHead(nn.Module):
    def __init__(self, in_features, emb_dim=EMB_DIM, num_classes=NUM_CLASSES, dropout=0.2, init_tau=16.0):
        super().__init__()
        self.bottleneck = nn.Sequential(
            nn.Linear(in_features, emb_dim, bias=False),
            nn.LayerNorm(emb_dim),
            nn.SiLU(inplace=True),
            nn.Dropout(p=dropout),
        )
        self.classifier = CosineLinear(emb_dim, num_classes=num_classes, init_tau=init_tau)
    def forward(self, x_3w):
        B, W, D = x_3w.shape
        h = self.bottleneck(x_3w.reshape(B * W, D))
        h_norm = F.normalize(h, p=2, dim=-1)
        logits_3w = self.classifier(h_norm).view(B, W, -1)
        h_3w = h_norm.view(B, W, -1)
        conf_weights = F.softmax(logits_3w.max(dim=-1, keepdim=True)[0] * 2.0, dim=1)
        bag_logits = (logits_3w * conf_weights).sum(dim=1)
        bag_emb = F.normalize((h_3w * conf_weights).sum(dim=1), p=2, dim=-1)
        return bag_logits, bag_emb

def compute_effective_num_weights(class_counts, beta=0.99):
    counts = np.asarray(class_counts, dtype=np.float64)
    weights = (1.0 - beta) / (1.0 - np.power(beta, counts))
    weights = weights / np.sum(weights) * len(counts)
    return torch.tensor(weights, dtype=torch.float32)

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.99, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.smoothing = smoothing
        self.num_classes = len(class_counts)
        self.register_buffer("alpha", compute_effective_num_weights(class_counts, beta=beta))
    def forward(self, logits, targets, sample_w=None):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        with torch.no_grad():
            true_dist = torch.zeros_like(logits)
            true_dist.fill_(self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        focal_weight = torch.pow(1.0 - probs, self.gamma)
        per_sample = -torch.sum(focal_weight * true_dist * log_probs, dim=-1)
        loss = self.alpha[targets] * per_sample   # bobot kelas berdasarkan TARGET (bukan per kelas di dalam jumlah)
        if sample_w is None:
            return loss.mean()
        return (loss * sample_w).sum() / sample_w.sum()

def train_variant(variant_name, X_train_3w, X_test_3w, in_features):
    oof_probs = np.zeros((n_tr, NUM_CLASSES), dtype=np.float32)
    oof_emb = np.zeros((n_tr, EMB_DIM), dtype=np.float32)   # PERINGATAN: tiap fold beda ruang embedding
    test_probs = np.zeros((n_te, NUM_CLASSES), dtype=np.float32)
    test_emb_folds = np.zeros((N_FOLDS, n_te, EMB_DIM), dtype=np.float32)
    fold_f1s = []

    print(f"\ntraining_{variant_name} (dim={in_features}, {N_EPOCHS_HEAD} ep/fold)...")
    for k, fold in enumerate(fold_ids):
        t_f = time.time()
        torch.manual_seed(SEED + int(fold))
        np.random.seed(SEED + int(fold))

        tr_idx = np.where(folds != fold)[0]
        val_idx = np.where(folds == fold)[0]

        x_tr = torch.from_numpy(X_train_3w[tr_idx])
        y_tr = torch.from_numpy(y_target[tr_idx]).long()
        x_va = torch.from_numpy(X_train_3w[val_idx]).to(DEVICE)
        x_te = torch.from_numpy(X_test_3w).to(DEVICE)
        y_va = y_target[val_idx]

        gen_dl = torch.Generator().manual_seed(SEED + int(fold))
        dl_tr = DataLoader(TensorDataset(x_tr, y_tr), batch_size=BATCH_SIZE_HEAD, shuffle=True, generator=gen_dl)

        model = PerchBottleneckCosineHead(in_features=in_features).to(DEVICE)
        decay, no_decay = [], []
        for _, p in model.named_parameters():
            (no_decay if p.ndim <= 1 else decay).append(p)
        optimizer = torch.optim.AdamW(
            [{"params": decay, "weight_decay": 1e-4}, {"params": no_decay, "weight_decay": 0.0}], lr=1e-3,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_HEAD, eta_min=1e-5)
        tr_counts = [int(np.sum(y_target[tr_idx] == c)) for c in range(NUM_CLASSES)]
        criterion = ClassBalancedFocalLoss(tr_counts).to(DEVICE)

        for ep in range(1, N_EPOCHS_HEAD + 1):
            model.train()
            for bx, by in dl_tr:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                optimizer.zero_grad()
                logits, _ = model(bx)
                criterion(logits, by).backward()
                optimizer.step()
            scheduler.step()

        model.eval()
        with torch.no_grad():
            v_logits, v_emb = model(x_va)
            v_probs = F.softmax(v_logits, dim=-1).cpu().numpy()
            v_emb_np = v_emb.cpu().numpy()
            t_logits, t_emb = model(x_te)
            t_probs = F.softmax(t_logits, dim=-1).cpu().numpy()
            t_emb_np = t_emb.cpu().numpy()

        v_pred = np.argmax(v_probs, axis=1)
        f1_val = float(f1_score(y_va, v_pred, average="macro"))
        fold_f1s.append(f1_val)
        cend_rec = float(recall_score(y_va == 1, v_pred == 1, zero_division=0))

        oof_probs[val_idx] = v_probs
        oof_emb[val_idx] = v_emb_np
        test_probs += t_probs / N_FOLDS
        test_emb_folds[k] = t_emb_np
        torch.save(model.state_dict(), WORK_DIR / f"head_{variant_name}_fold{int(fold)}.pt")

        print(f"  fold={k+1}/{N_FOLDS} | val={len(val_idx)} | macro_f1={f1_val:.4f} | "
              f"cendra_recall={cend_rec:.3f} (n={int((y_va == 1).sum())}) | {time.time()-t_f:.1f}s")
        del model, optimizer, scheduler, dl_tr, x_va, x_te, x_tr

    oof_pred = np.argmax(oof_probs, axis=1)
    global_f1 = float(f1_score(y_target, oof_pred, average="macro"))
    per_class = f1_score(y_target, oof_pred, average=None)
    cm = confusion_matrix(y_target, oof_pred, labels=[0, 1, 2])
    print(f"  {variant_name} Global OOF Macro F1 = {global_f1:.4f}")
    print(f"  Per-class: Bondol={per_class[0]:.4f} | Cendra={per_class[1]:.4f} | Kutilang={per_class[2]:.4f}")
    print(f"  CM:\n{cm}")

    purge_vram()
    return oof_probs, oof_emb, test_probs, test_emb_folds, global_f1, per_class

# --- Run A/B ---
train_hybrid = np.load(WORK_DIR / "train_hybrid_1543.npy")
test_hybrid = np.load(WORK_DIR / "test_hybrid_1543.npy")
oof_A, emb_A, test_A, temb_A, f1_A, pc_A = train_variant("hybrid_1543d", train_hybrid, test_hybrid, 1543)
oof_B, emb_B, test_B, temb_B, f1_B, pc_B = train_variant("pure_1536d", X_tr_emb, X_te_emb, 1536)

WINNER_MARGIN = 0.005   # satu seed: selisih kecil masih dalam noise, anggap seri -> pure
if f1_A > f1_B + WINNER_MARGIN:
    WINNER, IN_FEATURES = "hybrid_1543d", 1543
    oof_probs_sup, oof_emb_sup, test_probs_sup, test_emb_sup_folds = oof_A, emb_A, test_A, temb_A
else:
    WINNER, IN_FEATURES = "pure_1536d", 1536
    oof_probs_sup, oof_emb_sup, test_probs_sup, test_emb_sup_folds = oof_B, emb_B, test_B, temb_B

print(f"RESOLVE A/B RESULT:")
print(f"  Variant A (hybrid 1543-d) : F1={f1_A:.4f}")
print(f"  Variant B (pure 1536-d)   : F1={f1_B:.4f}")
print(f"  Delta (A-B)               : {f1_A - f1_B:+.4f}")
print(f"  WINNER                    : {WINNER}")

np.save(WORK_DIR / "oof_p_supervised.npy", oof_probs_sup)
np.save(WORK_DIR / "test_p_supervised.npy", test_probs_sup)
np.save(WORK_DIR / "oof_emb_supervised_256d.npy", oof_emb_sup)                     # tiap fold beda ruang, jangan dipakai lintas fold
np.save(WORK_DIR / "test_emb_supervised_256d_per_fold.npy", test_emb_sup_folds)    # valid per fold
np.save(WORK_DIR / "test_emb_supervised_256d.npy", test_emb_sup_folds.mean(axis=0))  # rata-rata lintas ruang, tidak bermakna

meta_c8 = {
    "variant_A_f1": f1_A, "variant_A_per_class": pc_A.tolist(),
    "variant_B_f1": f1_B, "variant_B_per_class": pc_B.tolist(),
    "winner": WINNER, "winner_in_features": IN_FEATURES,
    "loss_fix": "alpha_by_target",
    "emb_note": "256d embeddings from different folds are not in a shared space",
}
with open(WORK_DIR / "ab_test_result.json", "w") as fh:
    json.dump(meta_c8, fh, indent=2)

print(f"cell8_completed | winner={WINNER} | elapsed={time.time()-t0_c8:.1f}s")


training_hybrid_1543d (dim=1543, 40 ep/fold)...
  fold=1/4 | val=503 | macro_f1=0.9719 | cendra_recall=0.857 (n=35) | 10.2s
  fold=2/4 | val=462 | macro_f1=1.0000 | cendra_recall=1.000 (n=29) | 9.3s
  fold=3/4 | val=422 | macro_f1=0.8831 | cendra_recall=0.500 (n=16) | 9.4s
  fold=4/4 | val=439 | macro_f1=0.7301 | cendra_recall=1.000 (n=3) | 9.3s
  hybrid_1543d Global OOF Macro F1 = 0.9314
  Per-class: Bondol=0.9857 | Cendra=0.8092 | Kutilang=0.9992
  CM:
[[1104   20    0]
 [  12   70    1]
 [   0    0  619]]

training_pure_1536d (dim=1536, 40 ep/fold)...
  fold=1/4 | val=503 | macro_f1=0.9696 | cendra_recall=0.857 (n=35) | 8.9s
  fold=2/4 | val=462 | macro_f1=0.9983 | cendra_recall=1.000 (n=29) | 9.2s
  fold=3/4 | val=422 | macro_f1=0.8800 | cendra_recall=0.500 (n=16) | 9.1s
  fold=4/4 | val=439 | macro_f1=0.7003 | cendra_recall=0.667 (n=3) | 9.1s
  pure_1536d Global OOF Macro F1 = 0.9239
  Per-class: Bondol=0.9843 | Cendra=0.7931 | Kutilang=0.9944
  CM:
[[1099   22    3]
 [  10   69 

In [10]:
# Cell 9 - Nested In-Fold BirdNET Thresholds & Supervised Forensic Reassignment (revised)
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (roc_curve, confusion_matrix, f1_score,
                             precision_score, recall_score)

# ============ Bagian A: Ambang nested ============
t0_a = time.time()
LABELS = [0, 1, 2]

train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")
check(train_bnet.shape == (len(train_df), 3, 4), f"Shape tidak sejajar: {train_bnet.shape}")
check(bool(np.isfinite(train_bnet).all()), "NaN/Inf di BirdNET train")

cen_tr = train_bnet[:, :, 2].max(axis=1)
kut_tr = train_bnet[:, :, 1].max(axis=1)

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)

def youden_threshold(y_pos, scores):
    fpr, tpr, thr = roc_curve(y_pos.astype(int), scores)
    ok = np.isfinite(thr)
    j = int(np.argmax((tpr - fpr)[ok]))
    return float(thr[ok][j])

def rates(y_pos, scores, thr):
    pred = scores >= thr
    tpr = float(pred[y_pos].mean()) if y_pos.any() else np.nan
    fpr = float(pred[~y_pos].mean()) if (~y_pos).any() else np.nan
    return tpr, fpr

def compute_thresholds(mask):
    pos_c = y_target[mask] == 1
    pos_k = y_target[mask] == 2
    return {
        "cendra_p80": float(np.percentile(cen_tr[mask][~pos_c], 80.0)),
        "cendra_youden": youden_threshold(pos_c, cen_tr[mask]),
        "kutilang_p95": float(np.percentile(kut_tr[mask][~pos_k], 95.0)),
        "kutilang_youden": youden_threshold(pos_k, kut_tr[mask]),
    }

nested_thresholds = {}
print("computing_nested_in_fold_thresholds...")
print(f"{'fold':<5}{'ambang':<18}{'nilai':>8}   held-out TPR / FPR")
for f_id in fold_ids:
    tr_mask = folds != f_id
    va_mask = ~tr_mask
    thr = compute_thresholds(tr_mask)

    va_c = y_target[va_mask] == 1
    va_k = y_target[va_mask] == 2
    held = {}
    for name, thr_v in thr.items():
        if name.startswith("cendra"):
            held[name] = rates(va_c, cen_tr[va_mask], thr_v)
        else:
            held[name] = rates(va_k, kut_tr[va_mask], thr_v)
        print(f"{int(f_id):<5}{name:<18}{thr_v:>8.2f}   {held[name][0]:.3f} / {held[name][1]:.3f}")

    nested_thresholds[int(f_id)] = {
        "thr_cendra_p80": thr["cendra_p80"],
        "thr_cendra_youden": thr["cendra_youden"],
        "thr_kutilang_p95": thr["kutilang_p95"],
        "thr_kutilang_youden": thr["kutilang_youden"],
        "chosen_cendra": thr["cendra_p80"],
        "chosen_kutilang": thr["kutilang_p95"],
        "heldout_tpr_fpr": {k: [None if np.isnan(x) else x for x in v] for k, v in held.items()},
        "n_heldout_cendra": int(va_c.sum()),
        "n_heldout_kutilang": int(va_k.sum()),
    }

mean_thr_c = float(np.mean([v["chosen_cendra"] for v in nested_thresholds.values()]))
mean_thr_k = float(np.mean([v["chosen_kutilang"] for v in nested_thresholds.values()]))

final_thr = compute_thresholds(np.ones(len(train_df), dtype=bool))

meta_thr = {
    "fold_thresholds": nested_thresholds,
    "global_mean_thr_cendra": mean_thr_c,
    "global_mean_thr_kutilang": mean_thr_k,
    "final_thr_cendra": final_thr["cendra_p80"],
    "final_thr_kutilang": final_thr["kutilang_p95"],
    "final_all_methods": final_thr,
    "cendra_confidence_floor": 0.60,
    "note": ("ambang dihitung in-fold; pemilihan metode (p80/p95) berasal dari eksplorasi sebelumnya, "
             "jadi F1 sesudah reassignment optimistis. cendra_confidence_floor dan ambang Kutilang "
             "tidak dipakai di reassignment."),
}
with open(WORK_DIR / "nested_forensic_thresholds.json", "w") as fh:
    json.dump(meta_thr, fh, indent=2)

sd_c = float(np.std([v["chosen_cendra"] for v in nested_thresholds.values()]))
sd_k = float(np.std([v["chosen_kutilang"] for v in nested_thresholds.values()]))
print("\nnested_thresholds_locked:")
print(f"  Cendrawasih (p80) mean antar fold : {mean_thr_c:.2f} | final (seluruh train): {final_thr['cendra_p80']:.2f}")
print(f"  Kutilang    (p95) mean antar fold : {mean_thr_k:.2f} | final (seluruh train): {final_thr['kutilang_p95']:.2f}")
print(f"  Std antar fold: cendra={sd_c:.2f} | kutilang={sd_k:.2f}")
for name, val, lo, hi in [("Cendrawasih", mean_thr_c, -8.50, -7.00), ("Kutilang", mean_thr_k, -8.80, -7.50)]:
    if not (lo <= val <= hi):
        print(f"  [warn] Ambang {name} {val:.2f} di luar rentang acuan eksperimen lama [{lo}, {hi}]")
print(f"thresholds_saved -> {WORK_DIR / 'nested_forensic_thresholds.json'} | elapsed={time.time() - t0_a:.2f}s")

# ============ Bagian B: Forensic reassignment ============
t0_b = time.time()

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

oof_p = np.load(WORK_DIR / "oof_p_supervised.npy")
test_p = np.load(WORK_DIR / "test_p_supervised.npy")

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
fold_thrs = thr_meta["fold_thresholds"]
final_thr_cendra = thr_meta["final_thr_cendra"]

test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")

n_tr, n_te = len(train_df), len(sub_df)
check(oof_p.shape == (n_tr, 3) and test_p.shape == (n_te, 3), "Shape prediksi tidak sejajar")
check(train_bnet.shape[0] == n_tr and test_bnet.shape[0] == n_te, "Shape BirdNET tidak sejajar")

cen_oof = cen_tr
cen_test = test_bnet[:, :, 2].max(axis=1)

y_true = y_target
fold_keys = {str(int(f)) for f in fold_ids}
check(fold_keys <= set(fold_thrs), f"Kunci fold tidak ada di JSON: {fold_keys - set(fold_thrs)}")

def reassign(p, score, thr):
    """Jika prediksi = Cendrawasih tapi skor BirdNET < ambang, pindahkan massa Cendra ke kelas kedua."""
    pred_raw = np.argmax(p, axis=1)
    second = np.argsort(p, axis=1)[:, -2]
    mask = (pred_raw == 1) & (score < thr)
    corr = pred_raw.copy()
    corr[mask] = second[mask]
    p2 = p.copy()
    idx = np.where(mask)[0]
    p2[idx, second[idx]] += p2[idx, 1]
    p2[idx, 1] = 0.0
    p2 /= p2.sum(axis=1, keepdims=True)
    return pred_raw, corr, p2, mask

thr_oof = np.array([fold_thrs[str(int(f))]["chosen_cendra"] for f in folds])
pred_raw_oof, corr_pred_oof, oof_p_reassigned, mask_oof = reassign(oof_p, cen_oof, thr_oof)
oof_reassigned_count = int(mask_oof.sum())

f1_raw = float(f1_score(y_true, pred_raw_oof, average="macro", labels=LABELS))
f1_corr = float(f1_score(y_true, corr_pred_oof, average="macro", labels=LABELS))
f1_per_class_corr = f1_score(y_true, corr_pred_oof, average=None, labels=LABELS)
cm_raw = confusion_matrix(y_true, pred_raw_oof, labels=LABELS)
cm_corr = confusion_matrix(y_true, corr_pred_oof, labels=LABELS)

tp_lost = int((mask_oof & (y_true == 1)).sum())
fp_removed = int((mask_oof & (y_true != 1)).sum())
fp_fixed = int((mask_oof & (y_true != 1) & (corr_pred_oof == y_true)).sum())

rec_raw = float(recall_score(y_true == 1, pred_raw_oof == 1, zero_division=0))
rec_corr = float(recall_score(y_true == 1, corr_pred_oof == 1, zero_division=0))
prec_raw = float(precision_score(y_true == 1, pred_raw_oof == 1, zero_division=0))
prec_corr = float(precision_score(y_true == 1, corr_pred_oof == 1, zero_division=0))

print("\noof_forensic_reassignment_audit:")
print(f"  Dipindahkan dari Cendrawasih       : {oof_reassigned_count} (FP dibuang={fp_removed}, di antaranya benar={fp_fixed} | TP hilang={tp_lost})")
print(f"  Cendrawasih recall / precision     : {rec_raw:.3f} / {prec_raw:.3f}  ->  {rec_corr:.3f} / {prec_corr:.3f}")
print(f"  Macro F1 sebelum                   : {f1_raw:.4f}")
print(f"  Macro F1 sesudah                   : {f1_corr:.4f} (delta={f1_corr - f1_raw:+.4f})")
print(f"  Per-class F1 sesudah               : Bondol={f1_per_class_corr[0]:.4f} | Cendra={f1_per_class_corr[1]:.4f} | Kutilang={f1_per_class_corr[2]:.4f}")
print(f"Confusion Matrix sebelum (row=true, col=pred):\n{cm_raw}")
print(f"Confusion Matrix sesudah:\n{cm_corr}")

print("  Per fold (TP Cendra hilang / total Cendra | FP dibuang):")
for f_id in fold_ids:
    m = folds == f_id
    print(f"    fold {int(f_id)}: {int((mask_oof & m & (y_true == 1)).sum())}/{int((m & (y_true == 1)).sum())}"
          f" | {int((mask_oof & m & (y_true != 1)).sum())}")

pred_raw_test, corr_pred_test, test_p_reassigned, mask_test = reassign(test_p, cen_test, final_thr_cendra)
test_reassigned_count = int(mask_test.sum())

print("\ntest_anchor_reassignment_applied:")
print(f"  Test Cendrawasih calls raw         : {int((pred_raw_test == 1).sum())}")
print(f"  Test Cendrawasih calls reassigned  : {test_reassigned_count}")
print(f"  Test Cendrawasih calls remaining   : {int((corr_pred_test == 1).sum())}")
print(f"  Test class distribution raw        : {pd.Series([IDX_TO_CLASS[c] for c in pred_raw_test]).value_counts().to_dict()}")
print(f"  Test class distribution reassigned : {pd.Series([IDX_TO_CLASS[c] for c in corr_pred_test]).value_counts().to_dict()}")

np.save(WORK_DIR / "oof_p_supervised_reassigned.npy", oof_p_reassigned)
np.save(WORK_DIR / "test_p_supervised_anchor.npy", test_p_reassigned)

meta_c9 = {
    "oof_macro_f1_raw": f1_raw,
    "oof_macro_f1_reassigned": f1_corr,
    "oof_reassigned_samples": oof_reassigned_count,
    "oof_fp_removed": fp_removed,
    "oof_fp_fixed": fp_fixed,
    "oof_tp_lost": tp_lost,
    "oof_cendra_recall_raw": rec_raw,
    "oof_cendra_recall_reassigned": rec_corr,
    "oof_cendra_precision_raw": prec_raw,
    "oof_cendra_precision_reassigned": prec_corr,
    "test_cendra_raw": int((pred_raw_test == 1).sum()),
    "test_cendra_reassigned": test_reassigned_count,
    "test_cendra_clean": int((corr_pred_test == 1).sum()),
    "cm_raw": cm_raw.tolist(),
    "cm_reassigned": cm_corr.tolist(),
    "optimistic_note": "metode ambang p80 dipilih dari eksplorasi sebelumnya",
}
with open(WORK_DIR / "supervised_reassignment_meta.json", "w") as fh:
    json.dump(meta_c9, fh, indent=2)

print(f"\ncell9_completed | reassigned_oof_macro_f1={f1_corr:.4f} | elapsed={time.time() - t0_b:.2f}s")

computing_nested_in_fold_thresholds...
fold ambang               nilai   held-out TPR / FPR
0    cendra_p80           -7.66   0.886 / 0.171
0    cendra_youden        -7.34   0.886 / 0.145
0    kutilang_p95         -8.15   0.970 / 0.065
0    kutilang_youden      -7.37   0.955 / 0.016
1    cendra_p80           -7.70   1.000 / 0.201
1    cendra_youden        -7.34   0.966 / 0.171
1    kutilang_p95         -8.05   0.944 / 0.034
1    kutilang_youden      -7.34   0.916 / 0.006
2    cendra_p80           -7.68   0.812 / 0.185
2    cendra_youden        -7.24   0.688 / 0.131
2    kutilang_p95         -8.15   1.000 / 0.070
2    kutilang_youden      -7.37   1.000 / 0.016
3    cendra_p80           -7.77   1.000 / 0.239
3    cendra_youden        -7.25   1.000 / 0.154
3    kutilang_p95         -8.07   0.994 / 0.034
3    kutilang_youden      -7.37   0.994 / 0.011

nested_thresholds_locked:
  Cendrawasih (p80) mean antar fold : -7.70 | final (seluruh train): -7.70
  Kutilang    (p95) mean antar fold : 

In [11]:
# Cell 10 - kNN-5 Cosine Manifold Distance & Consensus Inlierness Curation (revised)
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

t0_c10 = time.time()
K_NEIGHBORS = 5
NUM_CLASSES = 3
P_PCT, D_PCT = 65.0, 50.0
KUT_CONF_FLOOR = 0.60

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat fitur & cek keselarasan
X_tr_emb = np.load(WORK_DIR / "perch_v2_train_emb.npy")
X_te_emb = np.load(WORK_DIR / "perch_v2_test_emb.npy")
D = X_tr_emb.shape[-1]
n_tr, n_te = len(train_df), len(sub_df)
check(X_tr_emb.shape[0] == n_tr and X_te_emb.shape[0] == n_te, "Embedding tidak sejajar dengan metadata")

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)

# 2. kNN-5 cosine in-fold (mean-centering dari train fold)
def center_norm(x, mu):
    x = x - mu
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def min_window_knn_dist(knn, emb_3w, mu):
    n, w, d = emb_3w.shape
    q = center_norm(emb_3w.reshape(n * w, d), mu)
    dists, _ = knn.kneighbors(q)
    return dists.mean(axis=1).reshape(n, w).min(axis=1)

oof_d_perch = np.zeros(n_tr, dtype=np.float32)
test_d_per_fold = np.zeros((N_FOLDS, n_te), dtype=np.float32)

print(f"computing_knn_distances (k={K_NEIGHBORS}, in-fold)...")
for k, fold in enumerate(fold_ids):
    tr_idx = np.where(folds != fold)[0]
    val_idx = np.where(folds == fold)[0]
    tr_w = X_tr_emb[tr_idx].reshape(-1, D)
    mu_fold = tr_w.mean(axis=0, keepdims=True)
    knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", n_jobs=-1).fit(center_norm(tr_w, mu_fold))
    oof_d_perch[val_idx] = min_window_knn_dist(knn, X_tr_emb[val_idx], mu_fold)
    test_d_per_fold[k] = min_window_knn_dist(knn, X_te_emb, mu_fold)

test_d_perch = test_d_per_fold.mean(axis=0).astype(np.float32)
check(bool(np.isfinite(oof_d_perch).all() and np.isfinite(test_d_perch).all()), "NaN/Inf pada jarak kNN")
np.save(WORK_DIR / "oof_d_perch.npy", oof_d_perch)
np.save(WORK_DIR / "test_d_perch.npy", test_d_perch)

print(f"  oof_manifold_distance  : mean={oof_d_perch.mean():.4f} | p50={np.percentile(oof_d_perch, 50):.4f} | p90={np.percentile(oof_d_perch, 90):.4f}")
print(f"  test_manifold_distance : mean={test_d_perch.mean():.4f} | p50={np.percentile(test_d_perch, 50):.4f} | p90={np.percentile(test_d_perch, 90):.4f}")
print("  oof_distance mean per kelas: " + " | ".join(
    f"{IDX_TO_CLASS[c]}={oof_d_perch[y_target == c].mean():.4f}" for c in range(NUM_CLASSES)))

# 3. Muat prediksi (setelah reassignment Cell 9), ambang, dan BirdNET
test_p_anchor = np.load(WORK_DIR / "test_p_supervised_anchor.npy")
oof_p_reassigned = np.load(WORK_DIR / "oof_p_supervised_reassigned.npy")
check(test_p_anchor.shape == (n_te, NUM_CLASSES) and oof_p_reassigned.shape == (n_tr, NUM_CLASSES),
      "Shape prediksi tidak sejajar")

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
final_thr_cendra = thr_meta["final_thr_cendra"]
final_thr_kutilang = thr_meta["final_thr_kutilang"]
fold_thrs = thr_meta["fold_thresholds"]

test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")
train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")
cen_test, kut_test = test_bnet[:, :, 2].max(axis=1), test_bnet[:, :, 1].max(axis=1)
cen_oof, kut_oof = train_bnet[:, :, 2].max(axis=1), train_bnet[:, :, 1].max(axis=1)

# Skor NLL dari prob kelas prediksi (bukan energi bebas; tidak dipakai untuk filter)
nll_maxp_test = -np.log(np.clip(test_p_anchor.max(axis=1), 1e-12, 1.0)).astype(np.float32)
np.save(WORK_DIR / "test_nll_maxp.npy", nll_maxp_test)

# 4. Aturan kurasi
def curate(p, d, cen, kut, thr_c, thr_k):
    pred = np.argmax(p, axis=1)
    maxp = p.max(axis=1)
    p_cut = float(np.percentile(maxp, P_PCT))
    d_cut = float(np.percentile(d, D_PCT))
    keep = (maxp >= p_cut) & (d <= d_cut)                                    # Filter 1: confidence & manifold
    keep &= ~((pred == 1) & (cen < thr_c))                                   # Filter 2: veto Cendrawasih
    keep &= ~((pred == 2) & ((kut < thr_k) | (maxp < KUT_CONF_FLOOR)))       # Filter 3: Kutilang shield
    return keep, pred, p_cut, d_cut

keep_te, pred_te, p_thresh, d_thresh = curate(
    test_p_anchor, test_d_perch, cen_test, kut_test, final_thr_cendra, final_thr_kutilang
)
pseudo_indices = np.where(keep_te)[0].astype(np.int64)
pseudo_targets = pred_te[keep_te].astype(np.int64)
dist_pseudo = {IDX_TO_CLASS[c]: int((pseudo_targets == c).sum()) for c in range(NUM_CLASSES)}

print("\nconsensus_inlierness_curation_locked:")
print(f"  Confidence cutoff (p{P_PCT:.0f})   : {p_thresh:.3f}")
print(f"  Manifold dist cutoff (p{D_PCT:.0f}): {d_thresh:.3f}")
print(f"  Curated pseudo-samples    : {len(pseudo_indices)}/{n_te} ({len(pseudo_indices) / n_te * 100:.1f}%)")
print(f"  Pseudo-label distribution : {dist_pseudo}")

# 5. Diagnostik OOF (batas atas: ambang metode dipilih dari eksplorasi sebelumnya)
thr_c_oof = np.array([fold_thrs[str(int(f))]["chosen_cendra"] for f in folds])
thr_k_oof = np.array([fold_thrs[str(int(f))]["chosen_kutilang"] for f in folds])
keep_oof, pred_oof, _, _ = curate(oof_p_reassigned, oof_d_perch, cen_oof, kut_oof, thr_c_oof, thr_k_oof)
purity = float((pred_oof[keep_oof] == y_target[keep_oof]).mean()) if keep_oof.any() else float("nan")
print(f"\noof_curation_diagnostic (aturan sama pada OOF; anggap batas atas):")
print(f"  Lolos kurasi : {int(keep_oof.sum())}/{n_tr} | kemurnian pseudo-label = {purity:.4f}")

purity_cls, retention_cls = {}, {}
for c in range(NUM_CLASSES):
    name = IDX_TO_CLASS[c]
    m = keep_oof & (pred_oof == c)
    pv = float((y_target[m] == c).mean()) if m.any() else None
    purity_cls[name] = {"n": int(m.sum()), "purity": pv}
    retention_cls[name] = float(keep_oof[y_target == c].mean())   # bagian sampel kelas asli yang lolos
    print(f"  {name:<12} n={int(m.sum()):<4} purity={'n/a' if pv is None else f'{pv:.4f}'} "
          f"| retention={retention_cls[name]:.3f} (dari {int((y_target == c).sum())} sampel asli)")
    if pv is not None and pv < 0.95:
        print(f"  [warn] Purity {name} {pv:.3f} < 0.95")

# 6. Simpan DULU, baru peringatan
np.save(WORK_DIR / "curated_pseudo_indices.npy", pseudo_indices)
np.save(WORK_DIR / "curated_pseudo_targets.npy", pseudo_targets)

meta_c10 = {
    "k_neighbors": K_NEIGHBORS,
    "oof_mean_dist": float(oof_d_perch.mean()),
    "test_mean_dist": float(test_d_perch.mean()),
    "p_thresh": p_thresh,
    "d_thresh": d_thresh,
    "p_percentile": P_PCT,
    "d_percentile": D_PCT,
    "n_curated": int(len(pseudo_indices)),
    "pseudo_distribution": dist_pseudo,
    "oof_curation_purity": purity,
    "oof_curation_purity_per_class": purity_cls,
    "oof_retention_per_class": retention_cls,
    "note": ("ambang persentil dihitung dari distribusi test (transduktif); skor energi tidak dipakai; "
             "purity OOF optimistis (ambang p80 dari eksplorasi sebelumnya)"),
}
with open(WORK_DIR / "curated_inlier_meta.json", "w") as fh:
    json.dump(meta_c10, fh, indent=2)

if not (100 <= len(pseudo_indices) <= 180):
    print(f"[warn] Jumlah pseudo-samples {len(pseudo_indices)} di luar rentang acuan [100, 180]")
for name, cnt in dist_pseudo.items():
    if cnt < 10:
        print(f"[warn] Pseudo-label {name} hanya {cnt} sampel")

print("artifacts_saved: curated_pseudo_indices.npy, curated_pseudo_targets.npy, oof_d_perch.npy, test_d_perch.npy, test_nll_maxp.npy")
print(f"cell10_completed | elapsed={time.time() - t0_c10:.2f}s")

computing_knn_distances (k=5, in-fold)...
  oof_manifold_distance  : mean=0.3315 | p50=0.3188 | p90=0.4837
  test_manifold_distance : mean=0.5110 | p50=0.5254 | p90=0.7054
  oof_distance mean per kelas: Bondol Jawa=0.3182 | Cendrawasih=0.4477 | Kutilang=0.3399

consensus_inlierness_curation_locked:
  Confidence cutoff (p65)   : 0.743
  Manifold dist cutoff (p50): 0.525
  Curated pseudo-samples    : 203/814 (24.9%)
  Pseudo-label distribution : {'Bondol Jawa': 92, 'Cendrawasih': 11, 'Kutilang': 100}

oof_curation_diagnostic (aturan sama pada OOF; anggap batas atas):
  Lolos kurasi : 352/1826 | kemurnian pseudo-label = 0.9972
  Bondol Jawa  n=259  purity=0.9961 | retention=0.230 (dari 1124 sampel asli)
  Cendrawasih  n=2    purity=1.0000 | retention=0.036 (dari 83 sampel asli)
  Kutilang     n=91   purity=1.0000 | retention=0.147 (dari 619 sampel asli)
[warn] Jumlah pseudo-samples 203 di luar rentang acuan [100, 180]
artifacts_saved: curated_pseudo_indices.npy, curated_pseudo_targets.npy

In [12]:
# Cell 11 - In-Fold Augmented Dataset Construction (Train 1543-d + Pseudo-Test per Fold) (revised)
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd

t0_c11 = time.time()
NUM_CLASSES = 3
CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat fitur hybrid & pseudo-label terkurasi (Cell 10)
train_hybrid = np.load(WORK_DIR / "train_hybrid_1543.npy")
test_hybrid = np.load(WORK_DIR / "test_hybrid_1543.npy")

pseudo_indices = np.load(WORK_DIR / "curated_pseudo_indices.npy")
pseudo_targets = np.load(WORK_DIR / "curated_pseudo_targets.npy")

n_tr, n_te = len(train_df), len(sub_df)
n_pseudo = len(pseudo_indices)
check(train_hybrid.shape == (n_tr, 3, 1543) and test_hybrid.shape == (n_te, 3, 1543), "Shape fitur hybrid tidak sejajar")
check(n_pseudo == len(pseudo_targets), "Panjang pseudo_indices dan pseudo_targets berbeda")
check(n_pseudo > 0, "Tidak ada pseudo-sample (jalankan ulang Cell 10)")
check(pseudo_indices.min() >= 0 and pseudo_indices.max() < n_te, "pseudo_indices di luar rentang test")
check(len(np.unique(pseudo_indices)) == n_pseudo, "pseudo_indices mengandung duplikat")
check(set(np.unique(pseudo_targets)) <= set(range(NUM_CLASSES)), "pseudo_targets di luar kelas 0..2")

pseudo_x = test_hybrid[pseudo_indices]
pseudo_y = pseudo_targets
pseudo_counts = [int((pseudo_y == c).sum()) for c in range(NUM_CLASSES)]
for c in range(NUM_CLASSES):
    if pseudo_counts[c] == 0:
        print(f"[warn] Tidak ada pseudo-sample untuk {IDX_TO_CLASS[c]}")

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)

# 2. Partisi per fold
augmented_fold_info = {}
print(f"building_in_fold_augmented_partitions (N_pseudo={n_pseudo}, per kelas={pseudo_counts})...")
for f in fold_ids:
    tr_mask = folds != f
    val_mask = folds == f
    n_tr_orig = int(tr_mask.sum())
    n_val_orig = int(val_mask.sum())

    orig_counts = [int((y_target[tr_mask] == c).sum()) for c in range(NUM_CLASSES)]
    aug_counts = [orig_counts[c] + pseudo_counts[c] for c in range(NUM_CLASSES)]
    boost_pct = [round(pseudo_counts[c] / max(orig_counts[c], 1) * 100, 1) for c in range(NUM_CLASSES)]

    augmented_fold_info[int(f)] = {
        "n_train_orig": n_tr_orig,
        "n_train_aug": n_tr_orig + n_pseudo,
        "n_val_pure_id": n_val_orig,
        "class_counts_orig": {IDX_TO_CLASS[c]: orig_counts[c] for c in range(NUM_CLASSES)},
        "class_counts_aug": {IDX_TO_CLASS[c]: aug_counts[c] for c in range(NUM_CLASSES)},
        "pseudo_boost_pct": {IDX_TO_CLASS[c]: boost_pct[c] for c in range(NUM_CLASSES)},
        "cendra_boost_samples": pseudo_counts[1],
        "cendra_boost_pct": boost_pct[1],
    }
    print(
        f"  fold={int(f)} | train_total={n_tr_orig + n_pseudo:<4} (orig={n_tr_orig} + pseudo={n_pseudo}) | "
        f"val_pure={n_val_orig:<3} | Cendra: {orig_counts[1]} -> {aug_counts[1]} (+{pseudo_counts[1]}, {boost_pct[1]}%)"
    )

# 3. Validasi keseluruhan
val_total_check = sum(v["n_val_pure_id"] for v in augmented_fold_info.values())
check(val_total_check == n_tr, f"Total sampel validasi {val_total_check} != {n_tr}")

# 4. Simpan metadata
meta_c11 = {
    "n_pseudo_total": int(n_pseudo),
    "pseudo_indices": [int(i) for i in pseudo_indices],
    "pseudo_distribution": {IDX_TO_CLASS[c]: pseudo_counts[c] for c in range(NUM_CLASSES)},
    "fold_partitions": augmented_fold_info,
    "leakage_note": ("pseudo-label berasal dari rata-rata 4 model fold Cell 8; 3 dari 4 model itu dilatih dengan "
                     "data fold validasi, jadi OOF setelah augmentasi optimistis dan tidak sebanding dengan Cell 8"),
}
with open(WORK_DIR / "augmented_in_fold_meta.json", "w") as fh:
    json.dump(meta_c11, fh, indent=2)

print(f"\nin_fold_augmented_ready | {N_FOLDS} folds configured | total_pseudo_injected={n_pseudo}")
print("  [note] OOF setelah augmentasi optimistis (pseudo-label dilihat dari model yang melatih fold validasi)")
print(f"saved_metadata -> {WORK_DIR / 'augmented_in_fold_meta.json'}")
print(f"cell11_completed | elapsed={time.time()-t0_c11:.2f}s")

building_in_fold_augmented_partitions (N_pseudo=203, per kelas=[92, 11, 100])...
  fold=0 | train_total=1526 (orig=1323 + pseudo=203) | val_pure=503 | Cendra: 48 -> 59 (+11, 22.9%)
  fold=1 | train_total=1567 (orig=1364 + pseudo=203) | val_pure=462 | Cendra: 54 -> 65 (+11, 20.4%)
  fold=2 | train_total=1607 (orig=1404 + pseudo=203) | val_pure=422 | Cendra: 67 -> 78 (+11, 16.4%)
  fold=3 | train_total=1590 (orig=1387 + pseudo=203) | val_pure=439 | Cendra: 80 -> 91 (+11, 13.8%)

in_fold_augmented_ready | 4 folds configured | total_pseudo_injected=203
  [note] OOF setelah augmentasi optimistis (pseudo-label dilihat dari model yang melatih fold validasi)
saved_metadata -> /kaggle/working/augmented_in_fold_meta.json
cell11_completed | elapsed=0.02s


In [13]:
# Cell 12 - Retraining 4-Fold Bottleneck Cosine Heads (Self-Training, fitur on-the-fly per fold)
import json, time, random
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

t0_c12 = time.time()

SEED = 42
N_EPOCHS_HEAD = 40
BATCH_SIZE_HEAD = 32
EMB_DIM = 256
NUM_CLASSES = 3
PSEUDO_W = 1.0   # coba 0.5 sebagai varian

for _v in ("WORK_DIR", "DEVICE", "train_df", "sub_df", "RESOLVE_GLOBAL_CENDRA_COL"):
    assert _v in globals(), f"Variabel global '{_v}' belum terdefinisi"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat data
with open(WORK_DIR / "perch_taxa_indices.json") as fh:
    taxa_meta = json.load(fh)
cendra_names = list(taxa_meta["cendra"].keys())
n_cand = len(cendra_names)

train_raw_sum = np.load(WORK_DIR / "train_raw_cand_summary.npy")
test_raw_sum = np.load(WORK_DIR / "test_raw_cand_summary.npy")
X_tr_emb = np.load(WORK_DIR / "perch_v2_train_emb.npy")
X_te_emb = np.load(WORK_DIR / "perch_v2_test_emb.npy")
oof_probs_base = np.load(WORK_DIR / "oof_p_supervised.npy")
test_probs_base = np.load(WORK_DIR / "test_p_supervised.npy")

pseudo_indices = np.load(WORK_DIR / "curated_pseudo_indices.npy")
pseudo_targets = np.load(WORK_DIR / "curated_pseudo_targets.npy").astype(np.int64)
n_pseudo = len(pseudo_indices)

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)
n_tr, n_te = len(train_df), len(sub_df)
IN_FEATURES = X_tr_emb.shape[-1] + 7

assert X_tr_emb.shape[0] == n_tr and X_te_emb.shape[0] == n_te, "Fitur tidak sejajar dengan metadata"
assert oof_probs_base.shape == (n_tr, NUM_CLASSES) and test_probs_base.shape == (n_te, NUM_CLASSES)
assert len(pseudo_targets) == n_pseudo and len(np.unique(pseudo_indices)) == n_pseudo, "Pseudo-label tidak valid"
assert n_pseudo == 0 or (pseudo_indices.min() >= 0 and pseudo_indices.max() < n_te), "Indeks pseudo di luar rentang"
assert set(np.unique(pseudo_targets)) <= set(range(NUM_CLASSES)), "Label pseudo tidak valid"
assert 0 <= RESOLVE_GLOBAL_CENDRA_COL < n_cand

# 2. Fitur (identik dengan Cell 7)
COL_B, COL_K = 0, 1
cendra_cols = list(range(2, 2 + n_cand))
COL_MAX_NT, COL_TOP5_NT, COL_ENERGY = 2 + n_cand, 3 + n_cand, 4 + n_cand

def build_summary_7d(raw, c_col):
    z_b, z_k = raw[:, :, COL_B:COL_B + 1], raw[:, :, COL_K:COL_K + 1]
    z_c = raw[:, :, c_col:c_col + 1]
    max_nt, top5_nt = raw[:, :, COL_MAX_NT:COL_MAX_NT + 1], raw[:, :, COL_TOP5_NT:COL_TOP5_NT + 1]
    energy = raw[:, :, COL_ENERGY:COL_ENERGY + 1]
    margin_ood = max_nt - np.maximum(np.maximum(z_b, z_k), z_c)
    return np.concatenate([z_b, z_c, z_k, max_nt, top5_nt, margin_ood, energy], axis=2).astype(np.float32)

def make_hybrid(raw, emb, c_col, mu, sd):
    return np.concatenate([emb, (build_summary_7d(raw, c_col) - mu) / sd], axis=-1).astype(np.float32)

# 3. Arsitektur
class CosineLinear(nn.Module):
    def __init__(self, in_features, num_classes=3, init_tau=16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype=torch.float32))

    def forward(self, x):
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(self.weight, p=2, dim=-1)
        return torch.clamp(self.tau, min=1.0, max=32.0) * F.linear(x_norm, w_norm)

class PerchBottleneckCosineHead(nn.Module):
    def __init__(self, in_features=IN_FEATURES, emb_dim=EMB_DIM, num_classes=NUM_CLASSES, dropout=0.2, init_tau=16.0):
        super().__init__()
        self.bottleneck = nn.Sequential(
            nn.Linear(in_features, emb_dim, bias=False),
            nn.LayerNorm(emb_dim),
            nn.SiLU(inplace=True),
            nn.Dropout(p=dropout),
        )
        self.classifier = CosineLinear(emb_dim, num_classes=num_classes, init_tau=init_tau)

    def forward(self, x_3w):
        B, W, D = x_3w.shape
        h = self.bottleneck(x_3w.reshape(B * W, D))
        h_norm = F.normalize(h, p=2, dim=-1)
        logits_3w = self.classifier(h_norm).view(B, W, -1)
        h_3w = h_norm.view(B, W, -1)
        conf_weights = F.softmax(logits_3w.max(dim=-1, keepdim=True)[0] * 2.0, dim=1)
        bag_logits = (logits_3w * conf_weights).sum(dim=1)
        bag_emb = F.normalize((h_3w * conf_weights).sum(dim=1), p=2, dim=-1)
        return bag_logits, bag_emb

# 4. Loss (alpha diindeks berdasarkan target; harus identik dengan Cell 8)
def compute_effective_num_weights(class_counts, beta=0.99):
    counts = np.asarray(class_counts, dtype=np.float64)
    weights = (1.0 - beta) / (1.0 - np.power(beta, counts))
    weights = weights / np.sum(weights) * len(counts)
    return torch.tensor(weights, dtype=torch.float32)

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.99, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma, self.smoothing, self.num_classes = gamma, smoothing, len(class_counts)
        self.register_buffer("alpha", compute_effective_num_weights(class_counts, beta=beta))

    def forward(self, logits, targets, sample_w=None):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        with torch.no_grad():
            true_dist = torch.full_like(logits, self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        focal_weight = torch.pow(1.0 - probs, self.gamma)
        per_sample = -torch.sum(focal_weight * true_dist * log_probs, dim=-1)
        loss = self.alpha[targets] * per_sample
        if sample_w is None:
            return loss.mean()
        return (loss * sample_w).sum() / sample_w.sum()

# 5. Training 4-fold
oof_probs_st = np.zeros((n_tr, NUM_CLASSES), dtype=np.float32)
oof_emb256_st = np.zeros((n_tr, EMB_DIM), dtype=np.float32)
test_probs_st = np.zeros((n_te, NUM_CLASSES), dtype=np.float32)
test_emb256_st_folds = np.zeros((N_FOLDS, n_te, EMB_DIM), dtype=np.float32)
fold_f1s_st, pseudo_keep_rates, pseudo_keep_cls = [], [], []
fold_species = {}

c_col = cendra_cols[RESOLVE_GLOBAL_CENDRA_COL]
if n_pseudo:
    agree_base = float((test_probs_base[pseudo_indices].argmax(1) == pseudo_targets).mean())
    print(f"pseudo vs baseline agreement: {agree_base:.3f} | pseudo class counts: "
          f"{[int((pseudo_targets == c).sum()) for c in range(NUM_CLASSES)]}")

print(f"retraining_self_trained_heads ({N_EPOCHS_HEAD} ep/fold, n_pseudo={n_pseudo}, pseudo_w={PSEUDO_W})...")
for k, fold in enumerate(fold_ids):
    t_f = time.time()
    torch.manual_seed(SEED + int(fold)); np.random.seed(SEED + int(fold)); torch.cuda.manual_seed_all(SEED + int(fold))

    tr_mask = folds != fold
    tr_idx, val_idx = np.where(tr_mask)[0], np.where(~tr_mask)[0]
    assert set(tr_idx).isdisjoint(set(val_idx))

    fold_species[int(fold)] = cendra_names[RESOLVE_GLOBAL_CENDRA_COL]
    s7_tr = build_summary_7d(train_raw_sum[tr_idx], c_col)
    mu = s7_tr.mean(axis=(0, 1), keepdims=True)
    sd = s7_tr.std(axis=(0, 1), keepdims=True) + 1e-6

    x_tr = make_hybrid(train_raw_sum[tr_idx], X_tr_emb[tr_idx], c_col, mu, sd)
    x_ps = make_hybrid(test_raw_sum[pseudo_indices], X_te_emb[pseudo_indices], c_col, mu, sd)
    x_va = make_hybrid(train_raw_sum[val_idx], X_tr_emb[val_idx], c_col, mu, sd)
    x_te = make_hybrid(test_raw_sum, X_te_emb, c_col, mu, sd)

    x_aug = np.concatenate([x_tr, x_ps], axis=0)
    y_aug = np.concatenate([y_target[tr_idx], pseudo_targets]).astype(np.int64)
    is_pseudo = np.concatenate([np.zeros(len(tr_idx), bool), np.ones(n_pseudo, bool)])
    y_va = y_target[val_idx]
    for arr in (x_aug, x_va, x_te):
        assert np.isfinite(arr).all()
    assert x_aug.shape[1:] == (3, IN_FEATURES)

    x_va_t = torch.from_numpy(x_va).to(DEVICE)
    x_te_t = torch.from_numpy(x_te).to(DEVICE)

    sample_w = np.where(is_pseudo, PSEUDO_W, 1.0).astype(np.float32)
    ds = TensorDataset(torch.from_numpy(x_aug), torch.from_numpy(y_aug), torch.from_numpy(sample_w))
    gen = torch.Generator().manual_seed(SEED + int(fold))
    dl_tr = DataLoader(ds, batch_size=BATCH_SIZE_HEAD, shuffle=True, generator=gen)

    model = PerchBottleneckCosineHead().to(DEVICE)
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-4}, {"params": no_decay, "weight_decay": 0.0}], lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_HEAD, eta_min=1e-5)

    aug_counts = [int(np.sum(y_aug == c)) for c in range(NUM_CLASSES)]
    criterion = ClassBalancedFocalLoss(aug_counts, beta=0.99, gamma=2.0).to(DEVICE)

    for ep in range(1, N_EPOCHS_HEAD + 1):
        model.train()
        for bx, by, bw in dl_tr:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(bx)
            loss = criterion(logits, by, bw)
            assert torch.isfinite(loss), f"Loss non-finite fold={fold} ep={ep}"
            loss.backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        v_logits, v_emb = model(x_va_t)
        v_probs = F.softmax(v_logits, dim=-1).cpu().numpy()
        v_emb = v_emb.cpu().numpy()
        t_logits, t_emb = model(x_te_t)
        t_probs = F.softmax(t_logits, dim=-1).cpu().numpy()
        t_emb = t_emb.cpu().numpy()

    v_pred = np.argmax(v_probs, axis=1)
    final_f1 = float(f1_score(y_va, v_pred, average="macro"))
    cendra_f1 = float(f1_score(y_va == 1, v_pred == 1, zero_division=0))
    fold_f1s_st.append(final_f1)

    if n_pseudo:
        ps_pred = t_probs[pseudo_indices].argmax(axis=1)
        keep_rate = float((ps_pred == pseudo_targets).mean())
        keep_cls = [float((ps_pred[pseudo_targets == c] == c).mean()) if (pseudo_targets == c).any() else float("nan")
                    for c in range(NUM_CLASSES)]
    else:
        keep_rate, keep_cls = float("nan"), [float("nan")] * NUM_CLASSES
    pseudo_keep_rates.append(keep_rate); pseudo_keep_cls.append(keep_cls)

    oof_probs_st[val_idx] = v_probs
    oof_emb256_st[val_idx] = v_emb
    test_probs_st += t_probs / N_FOLDS
    test_emb256_st_folds[k] = t_emb
    torch.save(model.state_dict(), WORK_DIR / f"head_selftrained_fold{int(fold)}.pt")

    print(f"  fold={k + 1}/{N_FOLDS} | train={len(x_aug):<4} (orig={len(tr_idx)}+pseudo={n_pseudo}) | val={len(val_idx):<4} | "
          f"macro_f1={final_f1:.4f} | cendra_f1={cendra_f1:.4f} | pseudo_fit={keep_rate:.3f} | {time.time() - t_f:.1f}s")

    del model, optimizer, scheduler, dl_tr, x_va_t, x_te_t, ds, x_aug, x_tr, x_ps, x_va, x_te
    torch.cuda.empty_cache()

# 6. Evaluasi OOF
assert np.isfinite(oof_probs_st).all() and np.isfinite(test_probs_st).all()
oof_preds_st = np.argmax(oof_probs_st, axis=1)
oof_preds_base = np.argmax(oof_probs_base, axis=1)
f1_st = f1_score(y_target, oof_preds_st, average="macro")
f1_base = f1_score(y_target, oof_preds_base, average="macro")
pc_st, pc_base = f1_score(y_target, oof_preds_st, average=None), f1_score(y_target, oof_preds_base, average=None)
rec_st, rec_base = recall_score(y_target, oof_preds_st, average=None), recall_score(y_target, oof_preds_base, average=None)
cm_st = confusion_matrix(y_target, oof_preds_st, labels=[0, 1, 2])

print("\nself_training_completed | OOF validasi = data train asli")
print(f"  Macro F1 baseline (oof_p_supervised; cek 0.9314 pra / 0.9504 pasca Cell 9): {f1_base:.4f}")
print(f"  Macro F1 self-trained : {f1_st:.4f} (delta={f1_st - f1_base:+.4f})")
print(f"  Mean per-fold F1      : {np.mean(fold_f1s_st):.4f} +/- {np.std(fold_f1s_st):.4f}")
print(f"Confusion Matrix (row=true, col=pred):\n{cm_st}")
for idx, name in IDX_TO_CLASS.items():
    print(f"  {name:<12} F1 {pc_base[idx]:.4f} -> {pc_st[idx]:.4f} ({pc_st[idx] - pc_base[idx]:+.4f}) | "
          f"recall {rec_base[idx]:.3f} -> {rec_st[idx]:.3f}")
print(f"  pseudo_fit mean: {np.nanmean(pseudo_keep_rates):.3f} | per kelas: "
      f"{np.round(np.nanmean(np.array(pseudo_keep_cls), axis=0), 3).tolist()}")
changed = int((np.argmax(test_probs_st, axis=1) != np.argmax(test_probs_base, axis=1)).sum())
print(f"  Prediksi test berubah vs supervised: {changed}/{n_te}")
print(f"  Test dist st  : {np.bincount(np.argmax(test_probs_st, axis=1), minlength=NUM_CLASSES).tolist()}")
print(f"  Test dist base: {np.bincount(np.argmax(test_probs_base, axis=1), minlength=NUM_CLASSES).tolist()}")

# 7. Simpan artefak
np.save(WORK_DIR / "oof_p_selftrained.npy", oof_probs_st)
np.save(WORK_DIR / "test_p_selftrained.npy", test_probs_st)
np.save(WORK_DIR / "oof_emb_selftrained_256d.npy", oof_emb256_st)
np.save(WORK_DIR / "test_emb_selftrained_256d_per_fold.npy", test_emb256_st_folds)
np.save(WORK_DIR / "test_emb_selftrained_256d.npy", test_emb256_st_folds.mean(axis=0))  # kompatibilitas; ruang antar fold tidak selaras
with open(WORK_DIR / "selftrained_fold_species.json", "w") as fh:
    json.dump(fold_species, fh, indent=2)

print("artifacts_saved: oof_p_selftrained.npy, test_p_selftrained.npy, test_emb_selftrained_256d_per_fold.npy")
print(f"cell12_completed | elapsed={time.time() - t0_c12:.1f}s")

pseudo vs baseline agreement: 0.936 | pseudo class counts: [92, 11, 100]
retraining_self_trained_heads (40 ep/fold, n_pseudo=203, pseudo_w=1.0)...
  fold=1/4 | train=1526 (orig=1323+pseudo=203) | val=503  | macro_f1=0.9523 | cendra_f1=0.8710 | pseudo_fit=1.000 | 12.6s
  fold=2/4 | train=1567 (orig=1364+pseudo=203) | val=462  | macro_f1=0.9936 | cendra_f1=0.9825 | pseudo_fit=1.000 | 12.3s
  fold=3/4 | train=1607 (orig=1404+pseudo=203) | val=422  | macro_f1=0.7882 | cendra_f1=0.4000 | pseudo_fit=1.000 | 12.8s
  fold=4/4 | train=1590 (orig=1387+pseudo=203) | val=439  | macro_f1=0.7584 | cendra_f1=0.2857 | pseudo_fit=1.000 | 12.4s

self_training_completed | OOF validasi = data train asli
  Macro F1 baseline (oof_p_supervised; cek 0.9314 pra / 0.9504 pasca Cell 9): 0.9314
  Macro F1 self-trained : 0.9356 (delta=+0.0042)
  Mean per-fold F1      : 0.8731 +/- 0.1014
Confusion Matrix (row=true, col=pred):
[[1119    3    2]
 [  20   60    3]
 [   0    0  619]]
  Bondol Jawa  F1 0.9857 -> 0.9890 

In [14]:
# Cell 13 - Self-Trained Forensic Reassignment & Adapted Test Matrix
import json, time
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

t0_c13 = time.time()
NUM_CLASSES = 3
KUT_CONF_FLOOR = 0.60

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

for _v in ("WORK_DIR", "train_df", "sub_df"):
    assert _v in globals(), f"Variabel global '{_v}' belum terdefinisi"

# 1. Muat artefak
oof_p_st = np.load(WORK_DIR / "oof_p_selftrained.npy")
test_p_st = np.load(WORK_DIR / "test_p_selftrained.npy")
oof_p_sup = np.load(WORK_DIR / "oof_p_supervised.npy")
test_p_sup = np.load(WORK_DIR / "test_p_supervised.npy")

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
fold_thrs = thr_meta["fold_thresholds"]
final_thr_cendra = thr_meta["final_thr_cendra"]
final_thr_kutilang = thr_meta["final_thr_kutilang"]

train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")
test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")

n_tr, n_te = len(train_df), len(sub_df)
assert oof_p_st.shape == (n_tr, NUM_CLASSES) and test_p_st.shape == (n_te, NUM_CLASSES), "Shape prediksi tidak sejajar"
assert oof_p_sup.shape == oof_p_st.shape and test_p_sup.shape == test_p_st.shape, "Shape baseline tidak sejajar"
assert train_bnet.shape[0] == n_tr and test_bnet.shape[0] == n_te, "Shape BirdNET tidak sejajar"
for _a in (oof_p_st, test_p_st, oof_p_sup, test_p_sup, train_bnet, test_bnet):
    assert np.isfinite(_a).all(), "NaN/Inf terdeteksi pada artefak input"

cen_oof, kut_oof = train_bnet[:, :, 2].max(axis=1), train_bnet[:, :, 1].max(axis=1)
cen_test, kut_test = test_bnet[:, :, 2].max(axis=1), test_bnet[:, :, 1].max(axis=1)

y_true = train_df["target"].values
folds = train_df["fold"].values
assert {str(int(f)) for f in np.unique(folds)} <= set(fold_thrs), "Kunci fold tidak ada di JSON"
for f in np.unique(folds):
    assert {"chosen_cendra", "chosen_kutilang"} <= set(fold_thrs[str(int(f))]), f"Key ambang fold {int(f)} tidak lengkap"

# 2. Reassignment (identik dengan Cell 9; idempoten)
def reassign(p, score, thr):
    pred_raw = np.argmax(p, axis=1)
    second = np.argsort(p, axis=1)[:, -2]
    mask = (pred_raw == 1) & (score < thr)
    corr = pred_raw.copy()
    corr[mask] = second[mask]
    p2 = p.copy()
    idx = np.where(mask)[0]
    p2[idx, second[idx]] += p2[idx, 1]
    p2[idx, 1] = 0.0
    p2 /= p2.sum(axis=1, keepdims=True)
    return pred_raw, corr, p2, mask

# 3. OOF
thr_c_oof = np.array([fold_thrs[str(int(f))]["chosen_cendra"] for f in folds])
thr_k_oof = np.array([fold_thrs[str(int(f))]["chosen_kutilang"] for f in folds])

pred_raw_st, corr_pred_st, oof_p_st_reassigned, mask_oof = reassign(oof_p_st, cen_oof, thr_c_oof)
oof_reassigned_st = int(mask_oof.sum())

f1_raw_st = float(f1_score(y_true, pred_raw_st, average="macro"))
f1_corr_st = float(f1_score(y_true, corr_pred_st, average="macro"))
f1_pc_corr_st = f1_score(y_true, corr_pred_st, average=None)
cm_raw_st = confusion_matrix(y_true, pred_raw_st, labels=[0, 1, 2])
cm_corr_st = confusion_matrix(y_true, corr_pred_st, labels=[0, 1, 2])

tp_lost = int((mask_oof & (y_true == 1)).sum())
fp_removed = int((mask_oof & (y_true != 1)).sum())
fp_fixed = int((mask_oof & (y_true != 1) & (corr_pred_st == y_true)).sum())
rec_raw = float(recall_score(y_true == 1, pred_raw_st == 1, zero_division=0))
rec_corr = float(recall_score(y_true == 1, corr_pred_st == 1, zero_division=0))
prec_raw = float(precision_score(y_true == 1, pred_raw_st == 1, zero_division=0))
prec_corr = float(precision_score(y_true == 1, corr_pred_st == 1, zero_division=0))

print("self_trained_oof_reassignment_audit:")
print(f"  Dipindahkan dari Cendrawasih  : {oof_reassigned_st} (FP dibuang={fp_removed}, benar={fp_fixed} | TP hilang={tp_lost})")
print(f"  Cendra recall / precision     : {rec_raw:.3f} / {prec_raw:.3f}  ->  {rec_corr:.3f} / {prec_corr:.3f}")
print(f"  Macro F1 sebelum              : {f1_raw_st:.4f}")
print(f"  Macro F1 sesudah              : {f1_corr_st:.4f} (delta={f1_corr_st - f1_raw_st:+.4f})")
print(f"  Per-class F1 sesudah          : Bondol={f1_pc_corr_st[0]:.4f} | Cendra={f1_pc_corr_st[1]:.4f} | Kutilang={f1_pc_corr_st[2]:.4f}")
print(f"Confusion Matrix sebelum (row=true, col=pred):\n{cm_raw_st}")
print(f"Confusion Matrix sesudah:\n{cm_corr_st}")

# Baseline supervised: dihitung ulang dengan ambang yang sama (apple-to-apple dengan Cell 9)
_, corr_sup, _, mask_sup = reassign(oof_p_sup, cen_oof, thr_c_oof)
f1_sup_reassigned = float(f1_score(y_true, corr_sup, average="macro"))
print(f"\n  Macro F1 supervised+reassign (dihitung ulang): {f1_sup_reassigned:.4f} | "
      f"self-trained+reassign: {f1_corr_st:.4f} (delta={f1_corr_st - f1_sup_reassigned:+.4f})")
print(f"  (cek: harus ~0.9504 jika Cell 9 sama; reassign supervised memindahkan {int(mask_sup.sum())} sampel)")

# 4. Trucuk Shield
def trucuk_flag(p, kut, thr_k):
    return (np.argmax(p, axis=1) == 2) & (p[:, 2] < KUT_CONF_FLOOR) & (kut < thr_k)

flag_oof = trucuk_flag(oof_p_st, kut_oof, thr_k_oof)
n_flag_oof = int(flag_oof.sum())
flag_true_kut = float((y_true[flag_oof] == 2).mean()) if n_flag_oof else float("nan")
print("\ntrucuk_shield_oof_diagnostic:")
print(f"  OOF ditandai: {n_flag_oof}/{n_tr} | dari itu Kutilang asli: {flag_true_kut:.3f}")
print("  (Jika mendekati 1, marker ini menandai Kutilang asli, bukan Alien Bulbul)")

pred_raw_test_st, corr_pred_test_st, test_p_st_adapted, mask_test = reassign(test_p_st, cen_test, final_thr_cendra)
test_reassigned_st_count = int(mask_test.sum())
trucuk_mask_test = trucuk_flag(test_p_st, kut_test, final_thr_kutilang) & ~mask_test

_, corr_pred_test_sup, _, _ = reassign(test_p_sup, cen_test, final_thr_cendra)
changed_vs_sup = int((corr_pred_test_st != corr_pred_test_sup).sum())

print("\ntest_self_trained_forensics_applied:")
print(f"  Test Cendrawasih calls raw         : {int((pred_raw_test_st == 1).sum())}")
print(f"  Test Cendrawasih calls reassigned  : {test_reassigned_st_count}")
print(f"  Test Cendrawasih calls clean       : {int((corr_pred_test_st == 1).sum())} (supervised+reassign: {int((corr_pred_test_sup == 1).sum())})")
print(f"  Trucuk shield ditandai (marker)    : {int(trucuk_mask_test.sum())} samples")
print(f"  Test distribution raw              : {pd.Series([IDX_TO_CLASS[c] for c in pred_raw_test_st]).value_counts().to_dict()}")
print(f"  Test distribution reassigned       : {pd.Series([IDX_TO_CLASS[c] for c in corr_pred_test_st]).value_counts().to_dict()}")
print(f"  Prediksi test berubah vs supervised+reassign: {changed_vs_sup}/{n_te}")

# 5. Simpan
np.save(WORK_DIR / "oof_p_selftrained_reassigned.npy", oof_p_st_reassigned)
np.save(WORK_DIR / "test_p_selftrained_adapted.npy", test_p_st_adapted)
np.save(WORK_DIR / "trucuk_mask_test.npy", trucuk_mask_test)

meta_c13 = {
    "oof_macro_f1_st_raw": f1_raw_st,
    "oof_macro_f1_st_reassigned": f1_corr_st,
    "oof_macro_f1_supervised_reassigned_recomputed": f1_sup_reassigned,
    "oof_reassigned_st_samples": oof_reassigned_st,
    "oof_fp_removed": fp_removed,
    "oof_fp_fixed": fp_fixed,
    "oof_tp_lost": tp_lost,
    "oof_cendra_recall": [rec_raw, rec_corr],
    "oof_cendra_precision": [prec_raw, prec_corr],
    "trucuk_oof_flagged": n_flag_oof,
    "trucuk_oof_flagged_true_kutilang_rate": None if n_flag_oof == 0 else flag_true_kut,
    "test_cendra_st_raw": int((pred_raw_test_st == 1).sum()),
    "test_cendra_st_reassigned": test_reassigned_st_count,
    "test_cendra_st_clean": int((corr_pred_test_st == 1).sum()),
    "test_changed_vs_supervised_reassigned": changed_vs_sup,
    "trucuk_shielded_test": int(trucuk_mask_test.sum()),
    "cm_raw_st": cm_raw_st.tolist(),
    "cm_reassigned_st": cm_corr_st.tolist(),
}
with open(WORK_DIR / "selftrained_reassignment_meta.json", "w") as fh:
    json.dump(meta_c13, fh, indent=2)

print(f"\ncell13_completed | final_self_trained_oof_f1={f1_corr_st:.4f} | elapsed={time.time() - t0_c13:.2f}s")

self_trained_oof_reassignment_audit:
  Dipindahkan dari Cendrawasih  : 3 (FP dibuang=2, benar=2 | TP hilang=1)
  Cendra recall / precision     : 0.723 / 0.952  ->  0.711 / 0.983
  Macro F1 sebelum              : 0.9356
  Macro F1 sesudah              : 0.9369 (delta=+0.0012)
  Per-class F1 sesudah          : Bondol=0.9894 | Cendra=0.8252 | Kutilang=0.9960
Confusion Matrix sebelum (row=true, col=pred):
[[1119    3    2]
 [  20   60    3]
 [   0    0  619]]
Confusion Matrix sesudah:
[[1121    1    2]
 [  21   59    3]
 [   0    0  619]]

  Macro F1 supervised+reassign (dihitung ulang): 0.9504 | self-trained+reassign: 0.9369 (delta=-0.0135)
  (cek: harus ~0.9504 jika Cell 9 sama; reassign supervised memindahkan 13 sampel)

trucuk_shield_oof_diagnostic:
  OOF ditandai: 4/1826 | dari itu Kutilang asli: 0.000
  (Jika mendekati 1, marker ini menandai Kutilang asli, bukan Alien Bulbul)

test_self_trained_forensics_applied:
  Test Cendrawasih calls raw         : 81
  Test Cendrawasih calls reas

In [15]:
# Cell 14 - Dual Forensic Taxonomic Shield (Supervised Anchor & Self-Trained Matrices)
import json, time
import numpy as np
import pandas as pd

t0_c14 = time.time()
NUM_CLASSES = 3
KUT_CONF_FLOOR = 0.60

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

for _v in ("WORK_DIR", "sub_df"):
    assert _v in globals(), f"Variabel global '{_v}' belum terdefinisi"
for _f in ("test_p_supervised_anchor.npy", "test_p_selftrained_adapted.npy", "trucuk_mask_test.npy",
           "nested_forensic_thresholds.json", "birdnet_test_summary.npy"):
    assert (WORK_DIR / _f).exists(), f"Artefak hilang: {_f}"

# 1. Muat artefak
p_sup_anchor = np.load(WORK_DIR / "test_p_supervised_anchor.npy")     # Cell 9
p_st_adapted = np.load(WORK_DIR / "test_p_selftrained_adapted.npy")   # Cell 13
trucuk_c13 = np.load(WORK_DIR / "trucuk_mask_test.npy")               # Cell 13 (definisi lama, pembanding)

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
final_thr_kutilang = thr_meta["final_thr_kutilang"]

test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")
kut_test = test_bnet[:, :, 1].max(axis=1)
n_te = len(sub_df)

assert p_sup_anchor.shape == (n_te, NUM_CLASSES) and p_st_adapted.shape == (n_te, NUM_CLASSES), "Shape tidak sejajar"
assert trucuk_c13.shape == (n_te,) and kut_test.shape == (n_te,)
for _a in (p_sup_anchor, p_st_adapted, kut_test):
    assert np.isfinite(_a).all(), "NaN/Inf terdeteksi"
assert np.allclose(p_sup_anchor.sum(1), 1.0, atol=1e-3) and np.allclose(p_st_adapted.sum(1), 1.0, atol=1e-3), \
    "Baris probabilitas tidak ternormalisasi"

# 2. Mask Trucuk (satu fungsi untuk kedua jalur)
def trucuk_flag(p, kut, thr_k):
    return (np.argmax(p, axis=1) == 2) & (p[:, 2] < KUT_CONF_FLOOR) & (kut < thr_k)

pred_sup = np.argmax(p_sup_anchor, axis=1)
pred_st = np.argmax(p_st_adapted, axis=1)
trucuk_mask_sup = trucuk_flag(p_sup_anchor, kut_test, final_thr_kutilang)
trucuk_mask_st = trucuk_flag(p_st_adapted, kut_test, final_thr_kutilang)

n_sup, n_st = int(trucuk_mask_sup.sum()), int(trucuk_mask_st.sum())
inter = int((trucuk_mask_sup & trucuk_mask_st).sum())
union = int((trucuk_mask_sup | trucuk_mask_st).sum())
jaccard = inter / union if union else float("nan")
diff_c13 = int((trucuk_mask_st != trucuk_c13).sum())

# 3. Perbandingan dua jalur
dist_sup = pd.Series([IDX_TO_CLASS[c] for c in pred_sup]).value_counts().to_dict()
dist_st = pd.Series([IDX_TO_CLASS[c] for c in pred_st]).value_counts().to_dict()
disagree_idx = np.where(pred_sup != pred_st)[0]
agreement = 1.0 - len(disagree_idx) / n_te
xtab = pd.crosstab(pd.Series([IDX_TO_CLASS[c] for c in pred_sup], name="supervised"),
                   pd.Series([IDX_TO_CLASS[c] for c in pred_st], name="self-trained"))

if len(disagree_idx):
    conf_sup_dis = float(p_sup_anchor[disagree_idx].max(1).mean())
    conf_st_dis = float(p_st_adapted[disagree_idx].max(1).mean())
    kut_dis = float(kut_test[disagree_idx].mean())
else:
    conf_sup_dis = conf_st_dis = kut_dis = float("nan")

print("dual_forensic_shield_summary:")
print(f"  Supervised  | Cendrawasih={dist_sup.get('Cendrawasih', 0)} | Trucuk ditandai={n_sup}")
print(f"  Self-Trained| Cendrawasih={dist_st.get('Cendrawasih', 0)} | Trucuk ditandai={n_st}")
print(f"  Overlap Trucuk: {inter} | dari sup={inter / max(n_sup, 1):.1%} | dari st={inter / max(n_st, 1):.1%} | Jaccard={jaccard:.3f}")
print(f"  Mask self-trained berbeda dari Cell 13 (definisi lama): {diff_c13} sampel")
print(f"  Supervised distribution   : {dist_sup}")
print(f"  Self-trained distribution : {dist_st}")
print(f"  Kesepakatan prediksi antar jalur: {agreement:.4f} ({len(disagree_idx)} berbeda dari {n_te})")
print(f"  Pada sampel berbeda | conf sup={conf_sup_dis:.3f} | conf st={conf_st_dis:.3f} | mean BirdNET Kut logit={kut_dis:.2f}")
print(f"Crosstab (baris=supervised, kolom=self-trained):\n{xtab}")

# 4. Simpan
np.save(WORK_DIR / "trucuk_mask_supervised.npy", trucuk_mask_sup)
np.save(WORK_DIR / "trucuk_mask_selftrained.npy", trucuk_mask_st)
np.save(WORK_DIR / "branch_disagreement_idx.npy", disagree_idx.astype(np.int64))

meta_c14 = {
    "supervised_cendra_clean": int(dist_sup.get("Cendrawasih", 0)),
    "selftrained_cendra_clean": int(dist_st.get("Cendrawasih", 0)),
    "supervised_trucuk_flagged": n_sup,
    "selftrained_trucuk_flagged": n_st,
    "trucuk_overlap": inter,
    "trucuk_jaccard": None if union == 0 else float(jaccard),
    "selftrained_mask_diff_vs_cell13": diff_c13,
    "branch_agreement": float(agreement),
    "n_branch_disagreement": int(len(disagree_idx)),
    "disagree_conf_sup": None if not len(disagree_idx) else conf_sup_dis,
    "disagree_conf_st": None if not len(disagree_idx) else conf_st_dis,
    "dist_supervised": {k: int(v) for k, v in dist_sup.items()},
    "dist_selftrained": {k: int(v) for k, v in dist_st.items()},
    "note": "mask Trucuk hanya marker (tidak mengubah prediksi); kedua jalur tidak independen karena pseudo-label berasal dari jalur supervised",
}
with open(WORK_DIR / "dual_forensics_meta.json", "w") as fh:
    json.dump(meta_c14, fh, indent=2)

print("\nartifacts_saved: trucuk_mask_supervised.npy, trucuk_mask_selftrained.npy, branch_disagreement_idx.npy, dual_forensics_meta.json")
print(f"cell14_completed | elapsed={time.time() - t0_c14:.2f}s")

dual_forensic_shield_summary:
  Supervised  | Cendrawasih=114 | Trucuk ditandai=66
  Self-Trained| Cendrawasih=75 | Trucuk ditandai=64
  Overlap Trucuk: 50 | dari sup=75.8% | dari st=78.1% | Jaccard=0.625
  Mask self-trained berbeda dari Cell 13 (definisi lama): 0 sampel
  Supervised distribution   : {'Bondol Jawa': 442, 'Kutilang': 258, 'Cendrawasih': 114}
  Self-trained distribution : {'Bondol Jawa': 486, 'Kutilang': 253, 'Cendrawasih': 75}
  Kesepakatan prediksi antar jalur: 0.9287 (58 berbeda dari 814)
  Pada sampel berbeda | conf sup=0.513 | conf st=0.504 | mean BirdNET Kut logit=-9.70
Crosstab (baris=supervised, kolom=self-trained):
self-trained  Bondol Jawa  Cendrawasih  Kutilang
supervised                                      
Bondol Jawa           438            0         4
Cendrawasih            33           75         6
Kutilang               15            0       243

artifacts_saved: trucuk_mask_supervised.npy, trucuk_mask_selftrained.npy, branch_disagreement_idx.npy, dual

In [16]:
# Cell 15 - Manifold Unknown Quantile Locking (laju Unknown = hyperparameter/prior, bukan hasil validasi OOF)
import json, time
import numpy as np

t0_c15 = time.time()
CLASS_NAMES = ["Bondol", "Cendra", "Kutilang"]

assert "WORK_DIR" in globals() and "sub_df" in globals(), "WORK_DIR/sub_df belum terdefinisi"
_files = ["oof_d_perch.npy", "test_d_perch.npy", "trucuk_mask_supervised.npy", "trucuk_mask_selftrained.npy",
          "test_p_supervised_anchor.npy", "test_p_selftrained_adapted.npy", "curated_pseudo_indices.npy"]
for _f in _files:
    assert (WORK_DIR / _f).exists(), f"Artefak hilang: {_f}"

# 1. Muat artefak
oof_d_perch = np.load(WORK_DIR / "oof_d_perch.npy")
test_d_perch = np.load(WORK_DIR / "test_d_perch.npy")
trucuk_mask_sup = np.load(WORK_DIR / "trucuk_mask_supervised.npy").astype(bool)
trucuk_mask_st = np.load(WORK_DIR / "trucuk_mask_selftrained.npy").astype(bool)
p_sup = np.load(WORK_DIR / "test_p_supervised_anchor.npy")
p_st = np.load(WORK_DIR / "test_p_selftrained_adapted.npy")
pseudo_idx = np.load(WORK_DIR / "curated_pseudo_indices.npy")

n_te = len(test_d_perch)
assert test_d_perch.ndim == 1 and oof_d_perch.ndim == 1, "Jarak harus 1-D"
assert len(sub_df) == n_te and len(trucuk_mask_sup) == n_te and len(trucuk_mask_st) == n_te, "Shape tidak sejajar"
assert p_sup.shape[0] == n_te and p_st.shape[0] == n_te, "Shape probabilitas tidak sejajar"
assert np.isfinite(test_d_perch).all() and np.isfinite(oof_d_perch).all(), "NaN/Inf pada jarak"
assert pseudo_idx.min() >= 0 and pseudo_idx.max() < n_te

# 2. Laju Unknown (prior; CV tidak bisa memvalidasi karena train tanpa Unknown)
RATE_STANDARD, RATE_PROBE_LOW, RATE_PROBE_HIGH = 0.4349, 0.4200, 0.4500
TARGET_STANDARD_COUNT = int(round(RATE_STANDARD * n_te))
TARGET_LOW_COUNT = int(round(RATE_PROBE_LOW * n_te))
TARGET_HIGH_COUNT = int(round(RATE_PROBE_HIGH * n_te))

def calibrate_tau(test_d, trucuk_mask, target_count):
    """Unknown = Trucuk-flag ATAU jarak terbesar; total persis target_count (berbasis ranking)."""
    cand = np.where(~trucuk_mask)[0]
    needed = target_count - int(trucuk_mask.sum())
    assert 0 < needed <= len(cand), (
        f"Trucuk={int(trucuk_mask.sum())} vs target={target_count}: needed={needed}, kandidat={len(cand)}")
    order = np.argsort(-test_d[cand], kind="stable")[:needed]
    is_unk = trucuk_mask.copy()
    is_unk[cand[order]] = True
    tau = float(test_d[cand[order]].min())                  # inklusif: d >= tau
    oof_reject = float((oof_d_perch >= tau).mean())         # indikatif saja (bank kNN OOF vs test berbeda)
    return tau, is_unk, int(is_unk.sum()), oof_reject, int(trucuk_mask.sum()), needed

runs = {
    "sub_019": ("ST RESOLVE @ 43.5%", trucuk_mask_st, TARGET_STANDARD_COUNT, p_st),
    "sub_020": ("ST RESOLVE @ 42.0%", trucuk_mask_st, TARGET_LOW_COUNT, p_st),
    "sub_021": ("Sup RESOLVE @ 43.5%", trucuk_mask_sup, TARGET_STANDARD_COUNT, p_sup),
    "sub_022": ("ST RESOLVE @ 45.0%", trucuk_mask_st, TARGET_HIGH_COUNT, p_st),
}
res = {}
print("unknown_quantiles_locked:")
for key, (desc, tmask, tgt, _) in runs.items():
    tau, is_unk, cnt, oof_rej, n_truc, n_dist = calibrate_tau(test_d_perch, tmask, tgt)
    assert cnt == tgt, f"{key}: jumlah Unknown {cnt} != target {tgt}"
    res[key] = dict(tau=tau, is_unk=is_unk, cnt=cnt, oof_rej=oof_rej, n_truc=n_truc, n_dist=n_dist)
    print(f"  {key} ({desc:<22}) tau>={tau:.4f} | Unknown={cnt}/{n_te} ({cnt / n_te:.2%}) | "
          f"Trucuk={n_truc}, jarak={n_dist} | OOF in-dist ditolak~{oof_rej:.1%} (indikatif)")

# 3. Diagnostik
u19, u20, u21, u22 = (res[k]["is_unk"] for k in ("sub_019", "sub_020", "sub_021", "sub_022"))
assert not (u20 & ~u19).any() and not (u19 & ~u22).any(), "Probe seharusnya bersarang (mask & ranking sama)"
print("\nunknown_diagnostics:")
print(f"  Selisih sampel: 019-020={int((u19 & ~u20).sum())} | 022-019={int((u22 & ~u19).sum())} "
      f"(~{0.3 * int((u19 & ~u20).sum()):.1f} sampel jatuh di Public LB: sinyal ~ noise)")
print(f"  sub_019 vs sub_021 mutual Unknown: {int((u19 & u21).sum())}/{res['sub_019']['cnt']}")
for name, p, u in (("ST ", p_st, u19), ("Sup", p_sup, u21)):
    pred = p.argmax(1)
    share = " | ".join(f"{CLASS_NAMES[c]}={int((u & (pred == c)).sum())}/{int((pred == c).sum())}" for c in range(p.shape[1]))
    print(f"  {name} Unknown per kelas prediksi: {share}")
print(f"  Pseudo-inlier yang ditandai Unknown: ST={int(u19[pseudo_idx].sum())}/{len(pseudo_idx)} | "
      f"Sup={int(u21[pseudo_idx].sum())}/{len(pseudo_idx)} (idealnya ~0)")

# 4. Simpan
for key in res:
    np.save(WORK_DIR / f"is_unk_{key}.npy", res[key]["is_unk"])

meta_c15 = {
    "rate_standard": RATE_STANDARD, "rate_probe_low": RATE_PROBE_LOW, "rate_probe_high": RATE_PROBE_HIGH,
    "overlap_sub19_sub21": int((u19 & u21).sum()),
    "pseudo_flagged_unknown_st": int(u19[pseudo_idx].sum()),
    "pseudo_flagged_unknown_sup": int(u21[pseudo_idx].sum()),
    "note": "laju Unknown = prior/hyperparameter (CV tidak dapat memvalidasi); probe 019/020/022 bersarang, selisih 12 sampel",
    **{k: {"tau_inclusive": v["tau"], "unknown_count": v["cnt"], "rate": v["cnt"] / n_te,
           "from_trucuk": v["n_truc"], "from_distance": v["n_dist"],
           "oof_indist_reject_rate_indicative": v["oof_rej"]} for k, v in res.items()},
}
with open(WORK_DIR / "quantile_locking_meta.json", "w") as fh:
    json.dump(meta_c15, fh, indent=2)

for k, v in res.items():
    if v["n_truc"] > 0.15 * v["cnt"]:
        print(f"[warn] {k}: {v['n_truc']} dari {v['cnt']} Unknown berasal dari flag Trucuk. Cek flag_true_kut di Cell 13.")

print("artifacts_saved: is_unk_sub_019.npy .. is_unk_sub_022.npy, quantile_locking_meta.json")
print(f"cell15_completed | elapsed={time.time() - t0_c15:.2f}s")

unknown_quantiles_locked:
  sub_019 (ST RESOLVE @ 43.5%    ) tau>=0.5872 | Unknown=354/814 (43.49%) | Trucuk=64, jarak=290 | OOF in-dist ditolak~5.6% (indikatif)
  sub_020 (ST RESOLVE @ 42.0%    ) tau>=0.5923 | Unknown=342/814 (42.01%) | Trucuk=64, jarak=278 | OOF in-dist ditolak~5.6% (indikatif)
  sub_021 (Sup RESOLVE @ 43.5%   ) tau>=0.5872 | Unknown=354/814 (43.49%) | Trucuk=66, jarak=288 | OOF in-dist ditolak~5.6% (indikatif)
  sub_022 (ST RESOLVE @ 45.0%    ) tau>=0.5784 | Unknown=366/814 (44.96%) | Trucuk=64, jarak=302 | OOF in-dist ditolak~5.9% (indikatif)

unknown_diagnostics:
  Selisih sampel: 019-020=12 | 022-019=12 (~3.6 sampel jatuh di Public LB: sinyal ~ noise)
  sub_019 vs sub_021 mutual Unknown: 349/354
  ST  Unknown per kelas prediksi: Bondol=230/486 | Cendra=19/75 | Kutilang=105/253
  Sup Unknown per kelas prediksi: Bondol=210/442 | Cendra=33/114 | Kutilang=111/258
  Pseudo-inlier yang ditandai Unknown: ST=0/203 | Sup=0/203 (idealnya ~0)
[warn] sub_019: 64 dari 354 Unk

In [18]:
# Cell 16 - Assembly, Verification, & Export Submissions (sub_019, sub_020, sub_021, sub_022)
import hashlib, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

t0_c16 = time.time()
VALID_CLASSES = ["Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"]
CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
CLASS_ARRAY = np.array([IDX_TO_CLASS[i] for i in range(len(IDX_TO_CLASS))], dtype=object)

DEFAULT_SUB = "sub_019"
SAMPLE_SUB_PATH = base_dir / "sample_submission.csv" if "base_dir" in globals() else None

for _v in ("WORK_DIR", "sub_df"):
    assert _v in globals(), f"Variabel global '{_v}' belum terdefinisi"
for _f in ("test_p_supervised_anchor.npy", "test_p_selftrained_adapted.npy",
           "is_unk_sub_019.npy", "is_unk_sub_020.npy", "is_unk_sub_021.npy", "is_unk_sub_022.npy"):
    assert (WORK_DIR / _f).exists(), f"Artefak hilang: {_f}"

# 1. Muat
p_sup = np.load(WORK_DIR / "test_p_supervised_anchor.npy")
p_st = np.load(WORK_DIR / "test_p_selftrained_adapted.npy")
masks = {k: np.load(WORK_DIR / f"is_unk_{k}.npy").astype(bool) for k in ("sub_019", "sub_020", "sub_021", "sub_022")}

n_te = len(sub_df)
test_fids = sub_df["file_id"].astype(str).values
for name, arr in (("p_sup", p_sup), ("p_st", p_st)):
    assert arr.shape == (n_te, 3), f"{name} shape {arr.shape} != ({n_te}, 3)"
    assert np.isfinite(arr).all(), f"{name} mengandung NaN/Inf"
for k, m in masks.items():
    assert m.shape == (n_te,), f"is_unk {k} shape {m.shape}"

# 2. Konfigurasi
CONFIGS = [
    ("sub_019", "experiment 19.csv", p_st,  "Day5_SelfTrained_GlobalTaxa_BirdNET_43.5pct", "Optimizer candidate: self-training @ 43.5% Unknown"),
    ("sub_020", "experiment 20.csv", p_st,  "Day5_SelfTrained_GlobalTaxa_BirdNET_42.0pct", "Probe low (nested, selisih ~12 sampel vs 019)"),
    ("sub_021", "experiment 21.csv", p_sup, "Day5_Supervised_GlobalTaxa_BirdNET_43.5pct",  "Anchor: supervised tanpa self-training @ 43.5%"),
    ("sub_022", "experiment 22.csv", p_st,  "Day5_SelfTrained_GlobalTaxa_BirdNET_45.0pct", "Probe high (nested, selisih ~12 sampel vs 019)"),
]
assert DEFAULT_SUB in {c[0] for c in CONFIGS}

def build_labels(p, unk):
    labels = CLASS_ARRAY[np.argmax(p, axis=1)].copy()
    labels[unk] = "Unknown"
    return labels

def validate(labels, unk, p, name):
    df = pd.DataFrame({"file_id": test_fids, "label": labels})
    assert len(df) == n_te and df["file_id"].nunique() == n_te, f"{name}: baris/file_id tidak valid"
    assert df.isna().sum().sum() == 0, f"{name}: NaN"
    bad = set(df["label"].unique()) - set(VALID_CLASSES)
    assert not bad, f"{name}: label ilegal {bad}"
    for c in VALID_CLASSES:
        assert (df["label"] == c).sum() > 0, f"{name}: kelas '{c}' 0 prediksi"
    assert int((labels == "Unknown").sum()) == int(unk.sum()), f"{name}: jumlah Unknown != mask"
    assert (labels[~unk] == CLASS_ARRAY[np.argmax(p, axis=1)][~unk]).all(), f"{name}: label non-Unknown berubah"
    return df

def md5_file(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

# 3. Verifikasi format terhadap sample_submission
if SAMPLE_SUB_PATH is not None and Path(SAMPLE_SUB_PATH).exists():
    ss = pd.read_csv(SAMPLE_SUB_PATH, dtype=str, keep_default_na=False)
    assert list(ss.columns) == ["file_id", "label"], f"Header sample_submission: {list(ss.columns)}"
    assert (ss["file_id"].values == test_fids).all(), "Urutan/isi file_id != sample_submission"
    print("sample_submission_format_check = OK")
else:
    print("[warn] SAMPLE_SUB_PATH belum terverifikasi")

# 4. Rakit + ekspor
all_labels, all_dfs, exported = {}, {}, {}
print("exporting_submissions:")
for sub_id, csv_name, p, strat, role in CONFIGS:
    labels = build_labels(p, masks[sub_id])
    df = validate(labels, masks[sub_id], p, sub_id)
    sub_dir = WORK_DIR / "submissions" / sub_id
    sub_dir.mkdir(parents=True, exist_ok=True)
    top = WORK_DIR / csv_name
    df.to_csv(top, index=False)
    df.to_csv(sub_dir / csv_name, index=False)

    chk = pd.read_csv(top, dtype=str, keep_default_na=False)
    assert list(chk.columns) == ["file_id", "label"], f"Header {csv_name} salah"
    assert (chk["file_id"].values == df["file_id"].values).all() and (chk["label"].values == df["label"].values).all(), \
        f"{csv_name} berubah setelah ekspor"

    dist = {k: int(v) for k, v in df["label"].value_counts().items()}
    md5v = md5_file(top)
    with open(sub_dir / "metrics.json", "w") as fh:
        json.dump({"submission_id": sub_id, "filename": csv_name, "strategy": strat, "scientific_role": role,
                   "n_samples": n_te, "unknown_percentage": round(float(np.mean(labels == "Unknown") * 100), 2),
                   "distribution": dist, "md5": md5v}, fh, indent=2)
    all_labels[sub_id], all_dfs[sub_id], exported[sub_id] = labels, df, top
    print(f"  {csv_name:<18} -> {dist} | Unk={np.mean(labels == 'Unknown') * 100:.2f}% | MD5={md5v[:8]}")
print("post_export_readback_check = OK")

# Default submission.csv
default_csv = next(c[1] for c in CONFIGS if c[0] == DEFAULT_SUB)
all_dfs[DEFAULT_SUB].to_csv(WORK_DIR / "submission.csv", index=False)
print(f"default_kaggle_submission = {DEFAULT_SUB} ({default_csv}) -> {WORK_DIR / 'submission.csv'}")

# 5. Kesepakatan antar submission (referensi: sub_019)
print("\nsubmission_cross_agreement_matrix (ref sub_019):")
for k in ("sub_020", "sub_021", "sub_022"):
    d = int(np.sum(all_labels["sub_019"] != all_labels[k]))
    print(f"  sub_019 vs {k}: {100 * (1 - d / n_te):.2f}% ({d} diffs)")
print("\nCrosstab sub_019 (baris) vs sub_021 (kolom):")
print(pd.crosstab(pd.Series(all_labels["sub_019"], name="sub_019"), pd.Series(all_labels["sub_021"], name="sub_021")))

# 6. Direct FileLink (tanpa tabel HTML)
print("\nDirect Download Links:")
cwd = Path.cwd().resolve()
for sub_id, csv_name, _, _, _ in CONFIGS:
    p = exported[sub_id]
    try:
        display(FileLink(str(p.resolve().relative_to(cwd))))
    except ValueError:
        display(FileLink(str(p.resolve())))

print(f"\ncell16_completed_day5 | {len(exported)}_submissions_exported | elapsed={time.time() - t0_c16:.2f}s")

sample_submission_format_check = OK
exporting_submissions:
  experiment 19.csv  -> {'Unknown': 354, 'Bondol Jawa': 256, 'Kutilang': 148, 'Cendrawasih': 56} | Unk=43.49% | MD5=b34660a8
  experiment 20.csv  -> {'Unknown': 342, 'Bondol Jawa': 266, 'Kutilang': 150, 'Cendrawasih': 56} | Unk=42.01% | MD5=ec4de6e2
  experiment 21.csv  -> {'Unknown': 354, 'Bondol Jawa': 232, 'Kutilang': 147, 'Cendrawasih': 81} | Unk=43.49% | MD5=8088df7b
  experiment 22.csv  -> {'Unknown': 366, 'Bondol Jawa': 249, 'Kutilang': 145, 'Cendrawasih': 54} | Unk=44.96% | MD5=93329651
post_export_readback_check = OK
default_kaggle_submission = sub_019 (experiment 19.csv) -> /kaggle/working/submission.csv

submission_cross_agreement_matrix (ref sub_019):
  sub_019 vs sub_020: 98.53% (12 diffs)
  sub_019 vs sub_021: 95.95% (33 diffs)
  sub_019 vs sub_022: 98.53% (12 diffs)

Crosstab sub_019 (baris) vs sub_021 (kolom):
sub_021      Bondol Jawa  Cendrawasih  Kutilang  Unknown
sub_019                                       

/kaggle/working/experiment 19.csv

/kaggle/working/experiment 20.csv

/kaggle/working/experiment 21.csv

/kaggle/working/experiment 22.csv


cell16_completed_day5 | 4_submissions_exported | elapsed=0.09s


In [20]:
# Cell 17 - Round-2 Self-Training (FEATURE_MODE: pooled vs global) -> experiment 23
import hashlib, json, time, random
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

t0_c17 = time.time()
SEED = 42
N_EPOCHS_HEAD = 40
BATCH_SIZE_HEAD = 32
EMB_DIM = 256
NUM_CLASSES = 3
PSEUDO_W = 0.85
KUT_CONF_FLOOR = 0.60
TARGET_UNKNOWN_RATE = 0.4496      # prior, bukan hasil validasi
P_PCT, D_PCT = 55.0, 60.0         # persentil kurasi Round 2
FEATURE_MODE = "pooled"           # "pooled" (max 8 taksa) atau "global" (RESOLVE lock) untuk ablasi

for _v in ("WORK_DIR", "DEVICE", "train_df", "sub_df"):
    assert _v in globals(), f"Variabel global '{_v}' belum terdefinisi"
if FEATURE_MODE == "global":
    assert "RESOLVE_GLOBAL_CENDRA_COL" in globals(), "RESOLVE_GLOBAL_CENDRA_COL belum terdefinisi"
assert FEATURE_MODE in ("pooled", "global")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
CLASS_ARRAY = np.array([IDX_TO_CLASS[i] for i in range(NUM_CLASSES)], dtype=object)

# 1. Muat
with open(WORK_DIR / "perch_taxa_indices.json") as fh:
    n_cand = len(json.load(fh)["cendra"])
cendra_cols = list(range(2, 2 + n_cand))
COL_MAX_NT, COL_TOP5_NT, COL_ENERGY = 2 + n_cand, 3 + n_cand, 4 + n_cand

train_raw_sum = np.load(WORK_DIR / "train_raw_cand_summary.npy")
test_raw_sum = np.load(WORK_DIR / "test_raw_cand_summary.npy")
X_tr_emb = np.load(WORK_DIR / "perch_v2_train_emb.npy")
X_te_emb = np.load(WORK_DIR / "perch_v2_test_emb.npy")
test_p_r1 = np.load(WORK_DIR / "test_p_selftrained_adapted.npy")
oof_r1_re = np.load(WORK_DIR / "oof_p_selftrained_reassigned.npy")
oof_sup = np.load(WORK_DIR / "oof_p_supervised.npy")
test_d_perch = np.load(WORK_DIR / "test_d_perch.npy")
test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")
train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")

cen_test, kut_test = test_bnet[:, :, 2].max(axis=1), test_bnet[:, :, 1].max(axis=1)
cen_oof = train_bnet[:, :, 2].max(axis=1)

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
final_thr_cendra, final_thr_kutilang = thr_meta["final_thr_cendra"], thr_meta["final_thr_kutilang"]
fold_thrs = thr_meta["fold_thresholds"]

n_tr, n_te = len(train_df), len(sub_df)
y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)
IN_FEATURES = X_tr_emb.shape[-1] + 7

assert train_raw_sum.shape[-1] >= COL_ENERGY + 1, "Kolom summary kurang dari ekspektasi"
assert test_p_r1.shape == (n_te, NUM_CLASSES) and test_d_perch.shape == (n_te,)
assert X_tr_emb.shape[0] == n_tr and X_te_emb.shape[0] == n_te
for _a in (test_p_r1, test_d_perch, X_tr_emb, X_te_emb, train_raw_sum, test_raw_sum, cen_test, kut_test):
    assert np.isfinite(_a).all(), "NaN/Inf pada input"
assert {str(int(f)) for f in fold_ids} <= set(fold_thrs)

# 2. Fitur
def build_summary_7d(raw):
    z_b, z_k = raw[:, :, 0:1], raw[:, :, 1:2]
    if FEATURE_MODE == "pooled":
        z_c = raw[:, :, cendra_cols].max(axis=2, keepdims=True)
    else:
        c = cendra_cols[RESOLVE_GLOBAL_CENDRA_COL]
        z_c = raw[:, :, c:c + 1]
    max_nt, top5_nt = raw[:, :, COL_MAX_NT:COL_MAX_NT + 1], raw[:, :, COL_TOP5_NT:COL_TOP5_NT + 1]
    energy = raw[:, :, COL_ENERGY:COL_ENERGY + 1]
    margin_ood = max_nt - np.maximum(np.maximum(z_b, z_k), z_c)
    return np.concatenate([z_b, z_c, z_k, max_nt, top5_nt, margin_ood, energy], axis=2).astype(np.float32)

def make_hybrid(raw, emb, mu, sd):
    return np.concatenate([emb, (build_summary_7d(raw) - mu) / sd], axis=-1).astype(np.float32)

# 3. Kurasi pseudo Round 2 (d_cut dijepit ke batas Unknown agar tidak overlap)
target_unk_count = int(round(TARGET_UNKNOWN_RATE * n_te))
pred_r1 = np.argmax(test_p_r1, axis=1)
maxp_r1 = test_p_r1.max(axis=1)
p_cut = float(np.percentile(maxp_r1, P_PCT))
d_unk_boundary = float(np.sort(test_d_perch)[n_te - target_unk_count - 1])   # di atas ini = kandidat Unknown (jarak)
d_cut = min(float(np.percentile(test_d_perch, D_PCT)), d_unk_boundary)

keep = (maxp_r1 >= p_cut) & (test_d_perch <= d_cut)
keep &= ~((pred_r1 == 1) & (cen_test < final_thr_cendra))
keep &= ~((pred_r1 == 2) & ((kut_test < final_thr_kutilang) | (maxp_r1 < KUT_CONF_FLOOR)))
pseudo_idx = np.where(keep)[0].astype(np.int64)
pseudo_targets = pred_r1[keep].astype(np.int64)
n_pseudo = len(pseudo_idx)
assert n_pseudo > 0, "Tidak ada pseudo-label lolos kurasi"
print(f"round2_pseudo | mode={FEATURE_MODE} | total={n_pseudo}/{n_te} | "
      f"dist={{{', '.join(f'{IDX_TO_CLASS[c]}: {int((pseudo_targets == c).sum())}' for c in range(NUM_CLASSES))}}} | "
      f"p_cut={p_cut:.3f} | d_cut={d_cut:.3f} (p{D_PCT:.0f}={np.percentile(test_d_perch, D_PCT):.3f})")

# 4. Arsitektur & loss (identik Cell 12 revisi)
class CosineLinear(nn.Module):
    def __init__(self, in_features, num_classes=3, init_tau=16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype=torch.float32))

    def forward(self, x):
        return torch.clamp(self.tau, min=1.0, max=32.0) * F.linear(F.normalize(x, p=2, dim=-1), F.normalize(self.weight, p=2, dim=-1))

class PerchBottleneckCosineHead(nn.Module):
    def __init__(self, in_features=IN_FEATURES, emb_dim=EMB_DIM, num_classes=NUM_CLASSES, dropout=0.2, init_tau=16.0):
        super().__init__()
        self.bottleneck = nn.Sequential(nn.Linear(in_features, emb_dim, bias=False), nn.LayerNorm(emb_dim),
                                        nn.SiLU(inplace=True), nn.Dropout(p=dropout))
        self.classifier = CosineLinear(emb_dim, num_classes=num_classes, init_tau=init_tau)

    def forward(self, x_3w):
        B, W, D = x_3w.shape
        h_norm = F.normalize(self.bottleneck(x_3w.reshape(B * W, D)), p=2, dim=-1)
        logits_3w = self.classifier(h_norm).view(B, W, -1)
        cw = F.softmax(logits_3w.max(dim=-1, keepdim=True)[0] * 2.0, dim=1)
        return (logits_3w * cw).sum(dim=1)

def effective_num_weights(counts, beta=0.99):
    counts = np.asarray(counts, dtype=np.float64)
    w = (1.0 - beta) / (1.0 - np.power(beta, counts))
    return torch.tensor(w / w.sum() * len(counts), dtype=torch.float32)

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.99, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma, self.smoothing, self.num_classes = gamma, smoothing, len(class_counts)
        self.register_buffer("alpha", effective_num_weights(class_counts, beta))

    def forward(self, logits, targets, sample_w=None):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        with torch.no_grad():
            td = torch.full_like(logits, self.smoothing / (self.num_classes - 1))
            td.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        per_sample = -torch.sum(torch.pow(1.0 - probs, self.gamma) * td * log_probs, dim=-1)
        loss = self.alpha[targets] * per_sample
        return loss.mean() if sample_w is None else (loss * sample_w).sum() / sample_w.sum()

# 5. Training 4-fold
oof_probs = np.zeros((n_tr, NUM_CLASSES), dtype=np.float32)
test_probs = np.zeros((n_te, NUM_CLASSES), dtype=np.float32)
fold_f1s = []

print(f"\ntraining_round2 ({N_EPOCHS_HEAD} ep/fold, n_pseudo={n_pseudo}, pseudo_w={PSEUDO_W})...")
for k, fold in enumerate(fold_ids):
    t_f = time.time()
    torch.manual_seed(SEED + int(fold)); np.random.seed(SEED + int(fold)); torch.cuda.manual_seed_all(SEED + int(fold))
    tr_idx, val_idx = np.where(folds != fold)[0], np.where(folds == fold)[0]
    assert set(tr_idx).isdisjoint(set(val_idx))

    s7_tr = build_summary_7d(train_raw_sum[tr_idx])
    mu, sd = s7_tr.mean(axis=(0, 1), keepdims=True), s7_tr.std(axis=(0, 1), keepdims=True) + 1e-6

    x_tr = make_hybrid(train_raw_sum[tr_idx], X_tr_emb[tr_idx], mu, sd)
    x_ps = make_hybrid(test_raw_sum[pseudo_idx], X_te_emb[pseudo_idx], mu, sd)
    x_va = make_hybrid(train_raw_sum[val_idx], X_tr_emb[val_idx], mu, sd)
    x_te = make_hybrid(test_raw_sum, X_te_emb, mu, sd)
    x_aug = np.concatenate([x_tr, x_ps], axis=0)
    y_aug = np.concatenate([y_target[tr_idx], pseudo_targets]).astype(np.int64)
    y_va = y_target[val_idx]
    for arr in (x_aug, x_va, x_te):
        assert np.isfinite(arr).all()
    assert x_aug.shape[1:] == (3, IN_FEATURES)

    x_va_t, x_te_t = torch.from_numpy(x_va).to(DEVICE), torch.from_numpy(x_te).to(DEVICE)
    sw = np.concatenate([np.ones(len(tr_idx)), np.full(n_pseudo, PSEUDO_W)]).astype(np.float32)
    ds = TensorDataset(torch.from_numpy(x_aug), torch.from_numpy(y_aug), torch.from_numpy(sw))
    dl = DataLoader(ds, batch_size=BATCH_SIZE_HEAD, shuffle=True, generator=torch.Generator().manual_seed(SEED + int(fold)))

    model = PerchBottleneckCosineHead().to(DEVICE)
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 1e-4}, {"params": no_decay, "weight_decay": 0.0}], lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS_HEAD, eta_min=1e-5)
    crit = ClassBalancedFocalLoss([int(np.sum(y_aug == c)) for c in range(NUM_CLASSES)]).to(DEVICE)

    for ep in range(1, N_EPOCHS_HEAD + 1):
        model.train()
        for bx, by, bw in dl:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(bx), by, bw)
            assert torch.isfinite(loss), f"Loss non-finite fold={fold} ep={ep}"
            loss.backward()
            opt.step()
        sched.step()

    model.eval()
    with torch.no_grad():
        v_probs = F.softmax(model(x_va_t), dim=-1).cpu().numpy()
        t_probs = F.softmax(model(x_te_t), dim=-1).cpu().numpy()
    v_pred = v_probs.argmax(1)
    f1_val = float(f1_score(y_va, v_pred, average="macro"))
    fold_f1s.append(f1_val)
    oof_probs[val_idx] = v_probs
    test_probs += t_probs / N_FOLDS
    print(f"  fold={k + 1}/{N_FOLDS} | val_f1={f1_val:.4f} | cendra_recall={recall_score(y_va == 1, v_pred == 1, zero_division=0):.3f} | {time.time() - t_f:.1f}s")
    del model, opt, sched, dl, ds, x_va_t, x_te_t, x_aug, x_tr, x_ps, x_va, x_te
    torch.cuda.empty_cache()

assert np.isfinite(oof_probs).all() and np.isfinite(test_probs).all()

# 6. Reassignment (OOF dengan ambang per fold; test dengan ambang final)
def reassign(p, score, thr):
    pred_raw = np.argmax(p, axis=1)
    second = np.argsort(p, axis=1)[:, -2]
    mask = (pred_raw == 1) & (score < thr)
    corr = pred_raw.copy(); corr[mask] = second[mask]
    p2 = p.copy(); idx = np.where(mask)[0]
    p2[idx, second[idx]] += p2[idx, 1]; p2[idx, 1] = 0.0
    p2 /= p2.sum(axis=1, keepdims=True)
    return pred_raw, corr, p2, mask

thr_c_oof = np.array([fold_thrs[str(int(f))]["chosen_cendra"] for f in folds])
_, oof_corr, _, _ = reassign(oof_probs, cen_oof, thr_c_oof)
f1_r2_raw = f1_score(y_target, oof_probs.argmax(1), average="macro")
f1_r2 = f1_score(y_target, oof_corr, average="macro")
f1_r1 = f1_score(y_target, oof_r1_re.argmax(1), average="macro")
_, sup_corr, _, _ = reassign(oof_sup, cen_oof, thr_c_oof)
f1_sup = f1_score(y_target, sup_corr, average="macro")
print(f"\nround2_oof (leakage orde-2 berlapis: delta hanya indikasi) | mode={FEATURE_MODE}")
print(f"  Macro F1 R2 raw={f1_r2_raw:.4f} | R2+reassign={f1_r2:.4f} | R1+reassign={f1_r1:.4f} (delta={f1_r2 - f1_r1:+.4f}) | "
      f"Sup+reassign={f1_sup:.4f} (delta={f1_r2 - f1_sup:+.4f})")
print(f"  Mean fold F1={np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}")
print(f"  Confusion Matrix R2+reassign:\n{confusion_matrix(y_target, oof_corr, labels=[0, 1, 2])}")

_, _, test_p_r2, mask_cendra = reassign(test_probs, cen_test, final_thr_cendra)
pred_r2 = test_p_r2.argmax(1)
trucuk = (pred_r2 == 2) & (test_p_r2[:, 2] < KUT_CONF_FLOOR) & (kut_test < final_thr_kutilang)

# 7. Kalibrasi Unknown
needed = target_unk_count - int(trucuk.sum())
cand = np.where(~trucuk)[0]
assert 0 < needed <= len(cand), f"Trucuk={int(trucuk.sum())} vs target={target_unk_count}: needed={needed}"
is_unk = trucuk.copy()
is_unk[cand[np.argsort(-test_d_perch[cand], kind="stable")[:needed]]] = True
assert int(is_unk.sum()) == target_unk_count

labels = CLASS_ARRAY[pred_r2].copy()
labels[is_unk] = "Unknown"
print(f"  Pseudo-inlier ditandai Unknown: {int(is_unk[pseudo_idx].sum())}/{n_pseudo} (harus ~0)")
print(f"  Test berubah vs R1: {int((pred_r2 != pred_r1).sum())}/{n_te} | Cendra reassigned: {int(mask_cendra.sum())} | Trucuk={int(trucuk.sum())}")

# 8. Export
df = pd.DataFrame({"file_id": sub_df["file_id"].astype(str).values, "label": labels})
assert len(df) == n_te and df["file_id"].nunique() == n_te and df["label"].isna().sum() == 0
assert set(df["label"].unique()) <= {"Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"}
tag = "" if FEATURE_MODE == "pooled" else "_global"
out_csv = WORK_DIR / f"experiment 23{tag}.csv"
df.to_csv(out_csv, index=False)
chk = pd.read_csv(out_csv, dtype=str, keep_default_na=False)
assert list(chk.columns) == ["file_id", "label"] and (chk["label"].values == df["label"].values).all()

np.save(WORK_DIR / f"oof_p_round2{tag}.npy", oof_probs)
np.save(WORK_DIR / f"test_p_round2_adapted{tag}.npy", test_p_r2)
np.save(WORK_DIR / f"is_unk_sub_023{tag}.npy", is_unk)
np.save(WORK_DIR / f"pseudo_idx_round2{tag}.npy", pseudo_idx)

ref = WORK_DIR / "experiment 19.csv"
if ref.exists():
    r19 = pd.read_csv(ref, dtype=str, keep_default_na=False)["label"].values
    print(f"  Beda vs experiment 19: {int((r19 != labels).sum())}/{n_te}")

md5 = hashlib.md5(out_csv.read_bytes()).hexdigest()
print(f"\nsub_023{tag}_exported | {out_csv.name} | MD5={md5[:8]} | {df['label'].value_counts().to_dict()} | Unk={np.mean(labels == 'Unknown') * 100:.2f}%")
print(f"cell17_completed | elapsed={time.time() - t0_c17:.1f}s")

from IPython.display import FileLink, display
display(FileLink(str(out_csv.resolve().relative_to(Path.cwd().resolve()))))


round2_pseudo | mode=pooled | total=274/814 | dist={Bondol Jawa: 143, Cendrawasih: 16, Kutilang: 115} | p_cut=0.730 | d_cut=0.558 (p60=0.587)

training_round2 (40 ep/fold, n_pseudo=274, pseudo_w=0.85)...
  fold=1/4 | val_f1=0.9531 | cendra_recall=0.771 | 12.4s
  fold=2/4 | val_f1=1.0000 | cendra_recall=1.000 | 12.7s
  fold=3/4 | val_f1=0.7628 | cendra_recall=0.188 | 12.5s
  fold=4/4 | val_f1=0.7749 | cendra_recall=0.333 | 13.1s

round2_oof (leakage orde-2 berlapis: delta hanya indikasi) | mode=pooled
  Macro F1 R2 raw=0.9386 | R2+reassign=0.9399 | R1+reassign=0.9369 (delta=+0.0030) | Sup+reassign=0.9504 (delta=-0.0105)
  Mean fold F1=0.8727 +/- 0.1052
  Confusion Matrix R2+reassign:
[[1124    0    0]
 [  22   59    2]
 [   0    0  619]]
  Pseudo-inlier ditandai Unknown: 0/274 (harus ~0)
  Test berubah vs R1: 27/814 | Cendra reassigned: 4 | Trucuk=68
  Beda vs experiment 19: 29/814

sub_023_exported | experiment 23.csv | MD5=bacc7e31 | {'Unknown': 366, 'Bondol Jawa': 254, 'Kutilang': 14

/kaggle/working/experiment 23.csv

In [21]:
# Cell 18 - Round-3 Iterative Transductive Self-Training (Deep Inlier Expansion) -> experiment 24
import hashlib, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import FileLink, display

t0_c18 = time.time()
SEED = 42
N_EPOCHS_HEAD = 40
BATCH_SIZE_HEAD = 32
EMB_DIM = 256
NUM_CLASSES = 3
PSEUDO_W = 0.80               # Sedikit diturunkan dari 0.85 untuk regularisasi volume inlier yang lebih besar
KUT_CONF_FLOOR = 0.60
TARGET_UNKNOWN_RATE = 0.4496   # Kunci sweet-spot terbukti sub_023 (366 sampel)
P_PCT, D_PCT = 50.0, 62.0      # Ekspansi adaptif kurasi Round 3

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
CLASS_ARRAY = np.array([IDX_TO_CLASS[i] for i in range(NUM_CLASSES)], dtype=object)

# 1. Muat data & artefak input
with open(WORK_DIR / "perch_taxa_indices.json") as fh:
    n_cand = len(json.load(fh)["cendra"])
cendra_cols = list(range(2, 2 + n_cand))
COL_MAX_NT, COL_TOP5_NT, COL_ENERGY = 2 + n_cand, 3 + n_cand, 4 + n_cand

train_raw_sum = np.load(WORK_DIR / "train_raw_cand_summary.npy")
test_raw_sum = np.load(WORK_DIR / "test_raw_cand_summary.npy")
X_tr_emb = np.load(WORK_DIR / "perch_v2_train_emb.npy")
X_te_emb = np.load(WORK_DIR / "perch_v2_test_emb.npy")

test_p_r2 = np.load(WORK_DIR / "test_p_round2_adapted.npy")   # Prediksi matang dari Round 2 (Cell 17)
test_d_perch = np.load(WORK_DIR / "test_d_perch.npy")
test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")
train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")

cen_test, kut_test = test_bnet[:, :, 2].max(axis=1), test_bnet[:, :, 1].max(axis=1)
cen_oof = train_bnet[:, :, 2].max(axis=1)

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
final_thr_cendra, final_thr_kutilang = thr_meta["final_thr_cendra"], thr_meta["final_thr_kutilang"]
fold_thrs = thr_meta["fold_thresholds"]

n_tr, n_te = len(train_df), len(sub_df)
y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)
IN_FEATURES = X_tr_emb.shape[-1] + 7

# 2. Fitur Multi-Taxa Paradisaeidae Pooling
def build_multitaxa_summary_7d(raw):
    z_b = raw[:, :, 0:1]
    z_k = raw[:, :, 1:2]
    z_c_pooled = raw[:, :, cendra_cols].max(axis=2, keepdims=True)
    max_nt = raw[:, :, COL_MAX_NT:COL_MAX_NT + 1]
    top5_nt = raw[:, :, COL_TOP5_NT:COL_TOP5_NT + 1]
    energy = raw[:, :, COL_ENERGY:COL_ENERGY + 1]
    margin_ood = max_nt - np.maximum(np.maximum(z_b, z_k), z_c_pooled)
    return np.concatenate([z_b, z_c_pooled, z_k, max_nt, top5_nt, margin_ood, energy], axis=2).astype(np.float32)

def make_multitaxa_hybrid(raw, emb, mu, sd):
    s7 = build_multitaxa_summary_7d(raw)
    return np.concatenate([emb, (s7 - mu) / sd], axis=-1).astype(np.float32)

# 3. Kurasi Pseudo-Labels Round 3 (Deep Inlier Expansion)
target_unk_count = int(round(TARGET_UNKNOWN_RATE * n_te))
pred_r2 = np.argmax(test_p_r2, axis=1)
maxp_r2 = test_p_r2.max(axis=1)

p_cut_r3 = float(np.percentile(maxp_r2, P_PCT))
d_unk_boundary = float(np.sort(test_d_perch)[n_te - target_unk_count - 1])
d_cut_r3 = min(float(np.percentile(test_d_perch, D_PCT)), d_unk_boundary)

keep_r3 = (maxp_r2 >= p_cut_r3) & (test_d_perch <= d_cut_r3)
keep_r3 &= ~((pred_r2 == 1) & (cen_test < final_thr_cendra))
keep_r3 &= ~((pred_r2 == 2) & ((kut_test < final_thr_kutilang) | (maxp_r2 < KUT_CONF_FLOOR)))

pseudo_idx_r3 = np.where(keep_r3)[0].astype(np.int64)
pseudo_targets_r3 = pred_r2[keep_r3].astype(np.int64)
n_pseudo_r3 = len(pseudo_idx_r3)
dist_r3 = {IDX_TO_CLASS[c]: int((pseudo_targets_r3 == c).sum()) for c in range(NUM_CLASSES)}

print(f"round3_pseudo_curation | total={n_pseudo_r3}/{n_te} ({n_pseudo_r3/n_te*100:.1f}%) | "
      f"dist={dist_r3} | p_cut={p_cut_r3:.3f} | d_cut={d_cut_r3:.3f}")

# 4. Arsitektur Model & Loss
class CosineLinear(nn.Module):
    def __init__(self, in_features, num_classes=3, init_tau=16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype=torch.float32))

    def forward(self, x):
        return torch.clamp(self.tau, min=1.0, max=32.0) * F.linear(F.normalize(x, p=2, dim=-1), F.normalize(self.weight, p=2, dim=-1))

class PerchBottleneckCosineHead(nn.Module):
    def __init__(self, in_features=IN_FEATURES, emb_dim=EMB_DIM, num_classes=NUM_CLASSES, dropout=0.2, init_tau=16.0):
        super().__init__()
        self.bottleneck = nn.Sequential(nn.Linear(in_features, emb_dim, bias=False), nn.LayerNorm(emb_dim),
                                        nn.SiLU(inplace=True), nn.Dropout(p=dropout))
        self.classifier = CosineLinear(emb_dim, num_classes=num_classes, init_tau=init_tau)

    def forward(self, x_3w):
        B, W, D = x_3w.shape
        h_norm = F.normalize(self.bottleneck(x_3w.reshape(B * W, D)), p=2, dim=-1)
        logits_3w = self.classifier(h_norm).view(B, W, -1)
        cw = F.softmax(logits_3w.max(dim=-1, keepdim=True)[0] * 2.0, dim=1)
        return (logits_3w * cw).sum(dim=1)

def effective_num_weights(counts, beta=0.99):
    counts = np.asarray(counts, dtype=np.float64)
    w = (1.0 - beta) / (1.0 - np.power(beta, counts))
    return torch.tensor(w / w.sum() * len(counts), dtype=torch.float32)

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.99, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma, self.smoothing, self.num_classes = gamma, smoothing, len(class_counts)
        self.register_buffer("alpha", effective_num_weights(class_counts, beta))

    def forward(self, logits, targets, sample_w=None):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        with torch.no_grad():
            td = torch.full_like(logits, self.smoothing / (self.num_classes - 1))
            td.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        per_sample = -torch.sum(torch.pow(1.0 - probs, self.gamma) * td * log_probs, dim=-1)
        loss = self.alpha[targets] * per_sample
        return loss.mean() if sample_w is None else (loss * sample_w).sum() / sample_w.sum()

# 5. Training 4-Fold Round 3
oof_probs_r3 = np.zeros((n_tr, NUM_CLASSES), dtype=np.float32)
test_probs_r3 = np.zeros((n_te, NUM_CLASSES), dtype=np.float32)
fold_f1s_r3 = []

print(f"\ntraining_round3 ({N_EPOCHS_HEAD} ep/fold, n_pseudo={n_pseudo_r3}, pseudo_w={PSEUDO_W})...")
for k, fold in enumerate(fold_ids):
    t_f = time.time()
    torch.manual_seed(SEED + int(fold)); np.random.seed(SEED + int(fold)); torch.cuda.manual_seed_all(SEED + int(fold))
    tr_idx, val_idx = np.where(folds != fold)[0], np.where(folds == fold)[0]

    s7_tr = build_multitaxa_summary_7d(train_raw_sum[tr_idx])
    mu, sd = s7_tr.mean(axis=(0, 1), keepdims=True), s7_tr.std(axis=(0, 1), keepdims=True) + 1e-6

    x_tr = make_multitaxa_hybrid(train_raw_sum[tr_idx], X_tr_emb[tr_idx], mu, sd)
    x_ps = make_multitaxa_hybrid(test_raw_sum[pseudo_idx_r3], X_te_emb[pseudo_idx_r3], mu, sd)
    x_va = make_multitaxa_hybrid(train_raw_sum[val_idx], X_tr_emb[val_idx], mu, sd)
    x_te = make_multitaxa_hybrid(test_raw_sum, X_te_emb, mu, sd)

    x_aug = np.concatenate([x_tr, x_ps], axis=0)
    y_aug = np.concatenate([y_target[tr_idx], pseudo_targets_r3]).astype(np.int64)
    y_va = y_target[val_idx]

    x_va_t, x_te_t = torch.from_numpy(x_va).to(DEVICE), torch.from_numpy(x_te).to(DEVICE)
    sw = np.concatenate([np.ones(len(tr_idx)), np.full(n_pseudo_r3, PSEUDO_W)]).astype(np.float32)
    ds = TensorDataset(torch.from_numpy(x_aug), torch.from_numpy(y_aug), torch.from_numpy(sw))
    dl = DataLoader(ds, batch_size=BATCH_SIZE_HEAD, shuffle=True, generator=torch.Generator().manual_seed(SEED + int(fold)))

    model = PerchBottleneckCosineHead().to(DEVICE)
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 1e-4}, {"params": no_decay, "weight_decay": 0.0}], lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS_HEAD, eta_min=1e-5)
    crit = ClassBalancedFocalLoss([int(np.sum(y_aug == c)) for c in range(NUM_CLASSES)]).to(DEVICE)

    for ep in range(1, N_EPOCHS_HEAD + 1):
        model.train()
        for bx, by, bw in dl:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(bx), by, bw)
            loss.backward()
            opt.step()
        sched.step()

    model.eval()
    with torch.no_grad():
        v_probs = F.softmax(model(x_va_t), dim=-1).cpu().numpy()
        t_probs = F.softmax(model(x_te_t), dim=-1).cpu().numpy()

    v_pred = v_probs.argmax(1)
    f1_val = float(f1_score(y_va, v_pred, average="macro"))
    cend_rec = float(recall_score(y_va == 1, v_pred == 1, zero_division=0))
    fold_f1s_r3.append(f1_val)

    oof_probs_r3[val_idx] = v_probs
    test_probs_r3 += t_probs / N_FOLDS
    print(f"  fold={k+1}/4 | val_f1={f1_val:.4f} | cendra_recall={cend_rec:.3f} | {time.time()-t_f:.1f}s")
    del model, opt, sched, dl, ds, x_va_t, x_te_t, x_aug, x_tr, x_ps, x_va, x_te
    torch.cuda.empty_cache()

# 6. Forensic Reassignment & Trucuk Shield
def reassign(p, score, thr):
    pred_raw = np.argmax(p, axis=1)
    second = np.argsort(p, axis=1)[:, -2]
    mask = (pred_raw == 1) & (score < thr)
    corr = pred_raw.copy(); corr[mask] = second[mask]
    p2 = p.copy(); idx = np.where(mask)[0]
    p2[idx, second[idx]] += p2[idx, 1]; p2[idx, 1] = 0.0
    p2 /= p2.sum(axis=1, keepdims=True)
    return pred_raw, corr, p2, mask

_, _, test_p_r3_reassigned, mask_cendra_r3 = reassign(test_probs_r3, cen_test, final_thr_cendra)
pred_r3 = test_p_r3_reassigned.argmax(1)

# Trucuk Shield
trucuk_mask_r3 = (pred_r3 == 2) & (test_p_r3_reassigned[:, 2] < KUT_CONF_FLOOR) & (kut_test < final_thr_kutilang)

# 7. Kalibrasi Unknown Sweet-Spot (366 sampel = 44.96%)
needed_r3 = target_unk_count - int(trucuk_mask_r3.sum())
cand_r3 = np.where(~trucuk_mask_r3)[0]
is_unk_r3 = trucuk_mask_r3.copy()
is_unk_r3[cand_r3[np.argsort(-test_d_perch[cand_r3], kind="stable")[:needed_r3]]] = True

labels_sub_024 = CLASS_ARRAY[pred_r3].copy()
labels_sub_024[is_unk_r3] = "Unknown"

# 8. Export sub_024 (experiment 24.csv)
sub_024_df = pd.DataFrame({"file_id": sub_df["file_id"].astype(str).values, "label": labels_sub_024})
assert len(sub_024_df) == n_te and sub_024_df["label"].isna().sum() == 0

out_csv_r3 = WORK_DIR / "experiment 24.csv"
sub_024_df.to_csv(out_csv_r3, index=False)
dist_024 = sub_024_df["label"].value_counts().to_dict()
md5_024 = hashlib.md5(out_csv_r3.read_bytes()).hexdigest()

np.save(WORK_DIR / "oof_p_round3.npy", oof_probs_r3)
np.save(WORK_DIR / "test_p_round3_adapted.npy", test_p_r3_reassigned)
np.save(WORK_DIR / "is_unk_sub_024.npy", is_unk_r3)

print(f"\nsub_024_exported (Round-3 Deep Inlier Model)")
print(f"  File        : experiment 24.csv")
print(f"  MD5         : {md5_024[:8]}")
print(f"  Distribution: {dist_024}")
print(f"  Unknown Rate: {np.mean(labels_sub_024 == 'Unknown')*100:.2f}%")
print(f"  Elapsed     : {time.time()-t0_c18:.1f}s")
display(FileLink("experiment 24.csv"))

round3_pseudo_curation | total=292/814 (35.9%) | dist={'Bondol Jawa': 156, 'Cendrawasih': 20, 'Kutilang': 116} | p_cut=0.738 | d_cut=0.558

training_round3 (40 ep/fold, n_pseudo=292, pseudo_w=0.8)...
  fold=1/4 | val_f1=0.9456 | cendra_recall=0.743 | 12.0s
  fold=2/4 | val_f1=1.0000 | cendra_recall=1.000 | 12.0s
  fold=3/4 | val_f1=0.7628 | cendra_recall=0.188 | 12.2s
  fold=4/4 | val_f1=0.7749 | cendra_recall=0.333 | 12.1s

sub_024_exported (Round-3 Deep Inlier Model)
  File        : experiment 24.csv
  MD5         : 1921b891
  Distribution: {'Unknown': 366, 'Bondol Jawa': 248, 'Kutilang': 146, 'Cendrawasih': 54}
  Unknown Rate: 44.96%
  Elapsed     : 48.2s


/kaggle/working/experiment 24.csv

In [22]:
# Cell 19 - Package & Zip All Day 5 Artifacts for Local Backup
import os, time, zipfile
from pathlib import Path
from datetime import datetime
from IPython.display import FileLink, display

t0_zip = time.time()
ts_str = datetime.now().strftime("%Y%m%d_%H%M")
work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
zip_filename = f"schematics_day5_all_artifacts_{ts_str}.zip"
zip_path = work_dir / zip_filename

valid_extensions = {".npy", ".pt", ".csv", ".json", ".png", ".txt", ".ipynb"}
ignored_prefixes = {"__pycache__", ".ipynb_checkpoints", ".git", "catboost_info", "birdnet_ckpt"}

print("archiving_all_day5_artifacts_to_zip...")
file_count = 0
total_uncompressed_bytes = 0

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zip_out:
    for root, dirs, files in os.walk(work_dir):
        dirs[:] = [d for d in dirs if not any(d.startswith(ign) for ign in ignored_prefixes)]
        for file in files:
            file_path = Path(root) / file
            if file == zip_filename or file.endswith(".zip") or file.endswith(".tmp.npy"):
                continue
            if any(file.startswith(ign) for ign in ignored_prefixes):
                continue
            if file_path.suffix.lower() in valid_extensions:
                rel_path = file_path.relative_to(work_dir)
                zip_out.write(file_path, arcname=str(rel_path))
                file_count += 1
                total_uncompressed_bytes += file_path.stat().st_size

zip_size_mb = zip_path.stat().st_size / (1024 * 1024)
raw_size_mb = total_uncompressed_bytes / (1024 * 1024)

print(f"DAY 5 ARTIFACTS ARCHIVE READY : {zip_filename}")
print(f"  total_files_archived : {file_count} files")
print(f"  uncompressed_size    : {raw_size_mb:.2f} MB")
print(f"  compressed_zip_size  : {zip_size_mb:.2f} MB (ratio: {raw_size_mb/max(0.1, zip_size_mb):.1f}x)")
print(f"  elapsed_time         : {time.time()-t0_zip:.2f}s")

static_zip = work_dir / "schematics_day5_all_artifacts.zip"
if zip_path != static_zip:
    import shutil
    shutil.copy2(zip_path, static_zip)

print("\nKlik link di bawah untuk unduh seluruh artefak Day 5:")
display(FileLink("schematics_day5_all_artifacts.zip"))

archiving_all_day5_artifacts_to_zip...
DAY 5 ARTIFACTS ARCHIVE READY : schematics_day5_all_artifacts_20261019_1958.zip
  total_files_archived : 82 files
  uncompressed_size    : 124.62 MB
  compressed_zip_size  : 114.66 MB (ratio: 1.1x)
  elapsed_time         : 6.93s

Klik link di bawah untuk unduh seluruh artefak Day 5:


/kaggle/working/schematics_day5_all_artifacts.zip